In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# !pip install -q -U "gptqmodel>=5.0"
# !pip install -q --force-reinstall --no-cache-dir "numpy==2.3.5" "scipy==1.16.2"
# !pip install -q tavily-python fastapi uvicorn pyngrok

In [ ]:
import os
import re
import json
import time
import threading
import torch
import gptqmodel
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient
from tavily import TavilyClient
from fastapi import FastAPI, HTTPException, Depends, Header
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok

In [ ]:
user_secrets = UserSecretsClient()

TAVILY_API_KEY = user_secrets.get_secret("TAVILY_API_KEY")
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
JOW_API_KEY = user_secrets.get_secret("JOW_API_KEY")

try:
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
except Exception:
    pass

print("Secrets loaded ✅")

In [ ]:
model_name = "Qwen/Qwen3-8B-AWQ"

tokenizer_qwen = AutoTokenizer.from_pretrained(model_name)

model_qwen = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Qwen loaded successfully ✅")

In [ ]:
generation_lock = threading.Lock()


def qwen_generate(messages, max_new_tokens=300, temperature=0.0, enable_thinking=False):
    import gc
    import torch

    prompt = tokenizer_qwen.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )
   
    # Limit prompt length to reduce GPU memory usage
    # (the cut removes the END of the prompt, so tell us when it happens)
    full_length = len(tokenizer_qwen(prompt)["input_ids"])
    if full_length > 4096:
        print(f"⚠️ Prompt cut: {full_length} tokens, limit is 4096")
    inputs = tokenizer_qwen(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model_qwen.device)
    
    if temperature > 0:
        sampling = dict(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        sampling = dict(do_sample=False)

    try:
        with generation_lock, torch.inference_mode():
            outputs = model_qwen.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer_qwen.eos_token_id,
                **sampling
            )

        new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
        result = tokenizer_qwen.decode(
            new_tokens, skip_special_tokens=True
        ).strip()

        if len(new_tokens) >= max_new_tokens:
            print(
                f"⚠️ Output cut: max_new_tokens={max_new_tokens}, "
                f"generated_tokens={len(new_tokens)}"
            )

        return result

    finally:
        del inputs
        if "outputs" in locals():
            del outputs
        if "new_tokens" in locals():
            del new_tokens
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

print("Tavily client ready ✅")

In [ ]:
def web_search(query, max_results=5, retries=3):
    for attempt in range(retries):
        try:
            response = tavily_client.search(
                query=query,
                search_depth="basic",
                max_results=max_results
            )
            return [
                {
                    "title": r.get("title", ""),
                    "url": r.get("url", ""),
                    "content": r.get("content", ""),
                    "score": r.get("score", 0)
                }
                for r in response.get("results", [])
            ]
        except Exception as e:
            print(f"⚠️ Tavily error ({attempt + 1}/{retries}): {e}")
            time.sleep(2)

    return []

In [ ]:
def parse_agent_result(result):
    # Already a dictionary
    if isinstance(result, dict):
        return result

    if not isinstance(result, str):
        return None

    # Keep only what is between the first { and the last }
    # (removes ```json fences and any text around the JSON)
    start = result.find("{")
    end = result.rfind("}")

    if start == -1 or end <= start:
        return None

    try:
        data = json.loads(result[start:end + 1])
    except json.JSONDecodeError:
        return None

    return data if isinstance(data, dict) else None

In [ ]:
def run_json_agent(name, messages, max_new_tokens, required_keys):
    raw = qwen_generate(messages, max_new_tokens=max_new_tokens, temperature=0.0)
    data = parse_agent_result(raw)

    if data is None:
        raise ValueError(f"{name} returned invalid JSON. Last 300 chars: {raw[-300:]}")

    missing = [k for k in required_keys if k not in data]
    if missing:
        raise ValueError(f"{name} JSON is missing keys: {missing}")

    return data


UNDERSTAND_PROMPT = """
You are the Idea Understanding Agent for "Jow El Tank",
an Egyptian startup evaluation platform.

Your job is to understand and structure the user's business idea
before the Sharks analyze it.

Use ONLY what the user wrote, plus what is obviously implied.

Do NOT add any detail or quality the user did not say.
Forbidden unless the user said it: cheap, affordable, healthy,
fast, high quality, expensive, competitive prices, or any
price, cost, commission, location, or technology.
Describe the need and the solution in neutral words only.

Do NOT evaluate the idea, research competitors,
give a success probability, or give financial estimates.

IMPORTANT for "business_model":
Fill it ONLY if the user explicitly says how the business makes money.
"Connecting" or "delivering" does NOT mean commission.
Otherwise write exactly: "مش محدد في الفكرة"

If another field cannot be determined, also write:
"مش محدد في الفكرة"

"missing_information" must contain only useful missing details
(including how the business earns money when it is not stated),
and must not contain "مش محدد في الفكرة".

Write in clear, simple Modern Standard Arabic.
(Egyptian dialect conversion is done later by another model.)
Keep every field short.

Example:

IDEA: تطبيق يتيح للمريض حجز موعد كشف عند الطبيب دون الانتظار في العيادة

OUTPUT:
{
  "problem": "يحتاج المريض إلى حجز موعد كشف عند الطبيب",
  "target_customer": "المرضى الذين يزورون الأطباء",
  "solution": "تطبيق يحجز للمريض موعد كشف محدد مع الطبيب",
  "business_idea": "تطبيق لحجز مواعيد الكشف عند الأطباء",
  "business_model": "مش محدد في الفكرة",
  "missing_information": [
    "كيف سيسجل الأطباء على التطبيق؟",
    "كيف سيحقق التطبيق الربح؟"
  ]
}

Return ONLY valid JSON with exactly these keys:

{
  "problem": "",
  "target_customer": "",
  "solution": "",
  "business_idea": "",
  "business_model": "",
  "missing_information": []
}
"""

UNDERSTAND_KEYS = [
    "problem", "target_customer", "solution",
    "business_idea", "business_model", "missing_information"
]


def understand_idea(idea):
    idea = idea.strip()

    messages = [
        {"role": "system", "content": UNDERSTAND_PROMPT},
        {"role": "user", "content": f"BUSINESS IDEA:\n{idea}"}
    ]

    data = run_json_agent("Idea Understanding", messages, 500, UNDERSTAND_KEYS)

    # missing_information must be a list
    if not isinstance(data["missing_information"], list):
        data["missing_information"] = [str(data["missing_information"])]

    return data

In [ ]:
EGYPT_WORDS = ["egypt", "cairo", "alexandria", "giza", "مصر", "القاهرة", "الإسكندرية", "الجيزة"]


def idea_to_search_topic(understanding):
    messages = [
        {
            "role": "system",
            "content": "Translate the business idea into a short English search phrase (max 10 words). Return only the phrase."
        },
        {
            "role": "user",
            "content": understanding["business_idea"] + ". " + understanding["solution"]
        }
    ]
    topic = qwen_generate(messages, max_new_tokens=30)
    topic = topic.strip().strip('"').split("\n")[0]
    return topic or understanding["business_idea"]


def mona_web_research(topic, understanding):
    queries = [
        f"Egypt competitors similar to {topic}",
        f"Egypt startups businesses similar to {topic}",
        f"global businesses similar to {topic}",
        f"Egypt market platforms related to {topic}",
        f"منصات {understanding['business_idea']} في مصر"
    ]

    all_results = []
    for query in queries:
        all_results.extend(web_search(query, max_results=5))

    return all_results


def prepare_mona_sources(research, max_sources=8, max_chars=1000):
    # Remove duplicate URLs
    unique = {}
    for r in research:
        if r["url"] and r["url"] not in unique:
            unique[r["url"]] = r

    # Keep the most relevant sources
    sources = sorted(unique.values(), key=lambda x: x["score"], reverse=True)[:max_sources]

    # Cut the content once, so the model and the checks see the same text
    sources = [{**r, "content": r["content"][:max_chars]} for r in sources]

    sources_text = ""
    for i, r in enumerate(sources, 1):
        sources_text += (
            f"SOURCE {i}\n"
            f"Title: {r['title']}\n"
            f"URL: {r['url']}\n"
            f"Content: {r['content']}\n\n"
        )

    return sources_text, sources


def valid_numbers(raw, sources):
    if not isinstance(raw, list):
        return []
    return [n for n in raw if isinstance(n, int) and 1 <= n <= len(sources)]


def source_text(numbers, sources):
    # Title + URL + content of the cited sources (lowercase, single spaces)
    text = " ".join(
        sources[n - 1]["title"] + " " + sources[n - 1]["url"] + " " + sources[n - 1]["content"]
        for n in numbers
    )
    return " ".join(text.lower().split())


def clean_competitors(items, competitor_type, sources):
    if not isinstance(items, list):
        raise ValueError(f"Mona: {competitor_type} list is not a list")

    kept, dropped = [], []

    for item in items:
        if not isinstance(item, dict):
            continue

        name = str(item.get("name", "")).strip()
        numbers = valid_numbers(item.get("source_numbers"), sources)
        text = source_text(numbers, sources)

        # The competitor must be named in the sources it cites
        if not name or not numbers or name.lower() not in text:
            dropped.append({"name": name, "reason": "name not found in the cited sources"})
            continue

        # Egyptian competitors must be shown to operate in Egypt
        if competitor_type != "global_business" and not any(w in text for w in EGYPT_WORDS):
            dropped.append({"name": name, "reason": "cited sources do not show it operates in Egypt"})
            continue

        kept.append({
            "name": name,
            "type": competitor_type,
            "reason": str(item.get("reason", "")),
            "source_numbers": numbers
        })

    return kept, dropped


def clean_notes(items, key, sources):
    if not isinstance(items, list):
        raise ValueError(f"Mona: {key} must be a list")

    kept, dropped = [], []

    for item in items:
        if not isinstance(item, dict):
            dropped.append({"name": str(item)[:60], "reason": f"{key}: item is not an object"})
            continue

        text = str(item.get("text", "")).strip()
        evidence = " ".join(str(item.get("evidence", "")).lower().split())
        numbers = valid_numbers(item.get("source_numbers"), sources)
        cited = source_text(numbers, sources)

        if not text or not evidence or not numbers:
            reason = "needs text, evidence and valid source_numbers"
        elif evidence not in cited:
            reason = "evidence quote not found in the cited sources"
        elif "مصر" in text and not any(w in evidence for w in EGYPT_WORDS):
            reason = "text mentions Egypt but the evidence quote does not"
        elif key == "market_gaps" and any(w in cited for w in EGYPT_WORDS):
            reason = "a source about activity in Egypt cannot prove a gap"
        else:
            kept.append({"text": text, "evidence": evidence, "source_numbers": numbers})
            continue

        dropped.append({"name": text[:60], "reason": f"{key}: {reason}"})

    return kept, dropped

def clean_advantages(items, idea_text):
    if not isinstance(items, list):
        raise ValueError("Mona: competitive_advantages must be a list")

    kept, dropped = [], []

    for item in items:
        text = str(item.get("text", "")).strip() if isinstance(item, dict) else ""
        evidence = " ".join(str(item.get("evidence", "")).lower().split()) if isinstance(item, dict) else ""

        if text and evidence and evidence in idea_text:
            kept.append({"text": text, "evidence": evidence, "source_numbers": []})
        else:
            dropped.append({
                "name": text[:60] or str(item)[:60],
                "reason": "competitive_advantages: evidence quote not found in the user's idea"
            })

    return kept, dropped
def build_verdict(direct, indirect, global_list):
    if direct:
        text = f"المصادر تثبت وجود {len(direct)} منافس مباشر في مصر."
    elif indirect:
        text = f"المصادر لا تثبت منافسًا مباشرًا في مصر، لكنها تثبت {len(indirect)} منافس غير مباشر."
    else:
        text = "المصادر لا تثبت وجود منافس مباشر أو غير مباشر في مصر، وهذا لا يعني غياب المنافسة."

    if global_list:
        text += f" ويوجد {len(global_list)} نموذج مشابه خارج مصر."

    return text


MONA_PROMPT = """
You are Mona, the Market & Competition Analyst
for an Egyptian startup evaluation platform.

Analyze the business idea using ONLY the WEB RESEARCH provided.

Rules:
1. Never invent companies, facts, statistics, or market claims.
2. direct_competitors: companies that the sources show operating IN EGYPT
   with a very similar product for the same customers.
   A source may describe an app in Egypt without naming it in the text:
   check the Title and the URL too, the company name may appear there.
3. indirect_competitors: companies that the sources show operating IN EGYPT
   and solving the same customer problem in a different way.
4. similar_global_businesses: companies OUTSIDE Egypt and the Arab world
   with a genuinely similar business model.
   A company outside the Arab world goes ONLY in this list.
5. Companies operating only in other Arab countries: do NOT list them,
   unless a source shows they operate in Egypt.
6. Never list generic categories (like "restaurants") as competitors.
7. Use the exact company name written in the sources, and put in
   source_numbers the SOURCE numbers that mention it.
8. "reason" must describe what THAT company does, based on its source,
   and where it operates. Do NOT copy the user's idea.
9.  market_gaps and market_risks:
   each item is {"text": "", "evidence": "", "source_numbers": []}.
   "evidence" is a short exact quote (max 15 words) copied word for word
   from the cited SOURCE content, in its original language.
   Do NOT translate it. The text must say only what the quote says.
   If there is no such quote, return [].
10. A market gap means a need that NO source shows being served.
    If a source describes a platform serving this need in Egypt,
    that platform is a competitor, never a gap.
11. competitive_advantages: each item is
    {"text": "", "evidence": "", "source_numbers": []}.
    "evidence" must be a short exact quote copied from the BUSINESS IDEA
    or the UNDERSTANDING (NOT from the sources), and source_numbers must be [].
    Never copy a trait of a competitor into the idea.
    If the idea states no differentiator, return [].
12. If you find only 1 or 2 competitors, return only those.
    Accuracy is more important than completeness.
13. Maximum 3 direct, 3 indirect, 2 global, 2 gaps,
    2 advantages, 2 risks. Keep every text to one short sentence.
14. "market_overview": one sentence describing which kinds of platforms
    the sources show. Do NOT repeat the idea. Do NOT mention speed,
    price, quality, women, or any opinion or opportunity.
15. Write text in clear, simple Modern Standard Arabic.
    Keep company names exactly as they appear in the sources.

Return ONLY valid JSON with exactly these keys and no extra nesting:

{
  "market_overview": "",
  "direct_competitors": [
    {"name": "", "type": "direct_competitor", "reason": "", "source_numbers": []}
  ],
  "indirect_competitors": [],
  "similar_global_businesses": [],
  "market_gaps": [
    {"text": "", "evidence": "", "source_numbers": []}
  ],
  "competitive_advantages": [],
  "market_risks": []
}
"""

MONA_KEYS = [
    "market_overview", "direct_competitors", "indirect_competitors",
    "similar_global_businesses", "market_gaps", "competitive_advantages",
    "market_risks"
]


def mona_market_analysis(idea, understanding):
    topic = idea_to_search_topic(understanding)
    research = mona_web_research(topic, understanding)
    sources_text, sources = prepare_mona_sources(research)

    if not sources:
        raise ValueError("Mona: Tavily returned no sources")

    messages = [
        {"role": "system", "content": MONA_PROMPT},
        {
            "role": "user",
            "content": (
                f"BUSINESS IDEA:\n{idea}\n\n"
                f"UNDERSTANDING:\n{json.dumps(understanding, ensure_ascii=False)}\n\n"
                f"WEB RESEARCH:\n{sources_text}"
            )
        }
    ]

    data = run_json_agent("Mona", messages, 1100, MONA_KEYS)

    result = {"market_overview": str(data["market_overview"])}
    dropped = []

    # Keep only competitors that are really in the sources (and in Egypt)
    for key, competitor_type in [
        ("direct_competitors", "direct_competitor"),
        ("indirect_competitors", "indirect_competitor"),
        ("similar_global_businesses", "global_business")
    ]:
        result[key], not_kept = clean_competitors(data[key], competitor_type, sources)
        dropped += not_kept

    # Keep only notes whose evidence quote is really in the sources
        # Gaps and risks must be proven by the sources
    for key in ["market_gaps", "market_risks"]:
        result[key], not_kept = clean_notes(data[key], key, sources)
        dropped += not_kept

    # Advantages must come from the user's own idea
    idea_text = " ".join(" ".join(str(v).lower().split()) for v in [idea, *understanding.values()])
    result["competitive_advantages"], not_kept = clean_advantages(data["competitive_advantages"], idea_text)
    dropped += not_kept

    # The verdict is built by Python from what was verified
    result["verdict"] = build_verdict(
        result["direct_competitors"],
        result["indirect_competitors"],
        result["similar_global_businesses"]
    )

    result["sources"] = [
        {"number": i, "title": r["title"], "url": r["url"]}
        for i, r in enumerate(sources, 1)
    ]
    result["dropped_items"] = dropped
    result["search_topic"] = topic

    return result

In [ ]:
def to_number(value, name):
    if isinstance(value, bool):
        raise ValueError(f"Mahmoud: {name} must be a number")

    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        try:
            return float(value.replace(",", "").strip())
        except ValueError:
            pass

    raise ValueError(f"Mahmoud: {name} must be a number, got: {value!r}")


def validate_assumptions(raw):
    if not isinstance(raw, dict):
        raise ValueError("Mahmoud: assumptions must be an object")

    price = to_number(raw.get("average_price"), "average_price")
    commission = to_number(raw.get("commission_rate"), "commission_rate")
    variable_cost = to_number(raw.get("variable_cost_per_order"), "variable_cost_per_order")
    fixed_costs = to_number(raw.get("fixed_costs"), "fixed_costs")

    # 15 means 15%
    if commission > 1:
        commission = commission / 100

    if price <= 0:
        raise ValueError("Mahmoud: average_price must be > 0")
    if not 0 < commission <= 1:
        raise ValueError("Mahmoud: commission_rate must be between 0 and 1")
    if variable_cost < 0 or fixed_costs < 0:
        raise ValueError("Mahmoud: costs cannot be negative")

    return {
        "average_price": price,
        "commission_rate": commission,
        "variable_cost_per_order": variable_cost,
        "fixed_costs": fixed_costs
    }


def clean_text_list(items, name, limit=3):
    if not isinstance(items, list):
        raise ValueError(f"Mahmoud: {name} must be a list")
    return [str(x).strip() for x in items if str(x).strip()][:limit]


MAHMOUD_PROMPT = """
You are Mahmoud, the Finance & Business Analyst
for an Egyptian startup evaluation platform.

Your ONLY job is to propose the financial ASSUMPTIONS of the idea.
Another program will do all the calculations.

Rules:
1. Do NOT calculate anything: no revenue, contribution, profit,
   break-even, or totals. Do not write calculated results in any text.
2. Give plain numbers only (no currency symbols, no text, no null).
3. All money values are in Egyptian pounds (EGP).
4. First use any finance-related answers explicitly provided by the founder
   in understanding.shared_interview_context.confirmed_facts. Never overwrite
   a founder-confirmed number with an estimate. For missing numbers only, use
   conservative estimates and label them as estimates in the explanation.
   Keep unresolved values in shared_interview_context.unknowns; do not present
   estimates as confirmed facts.
5. "business_model": one sentence, based on the understanding and confirmed answers.
   If the idea does not say how it earns money, start the sentence
   with "افتراض:" and use a simple commission on each order.
6. Assumptions (all numbers):
   - average_price: average order value paid by the customer
   - commission_rate: the share the platform keeps from each order,
     as a fraction (for example 0.15 means 15%)
   - variable_cost_per_order: the PLATFORM's own variable cost per order
     (payment fees, delivery cost paid by the platform, support).
     NOT the cost of making the food.
   - fixed_costs: the platform's total fixed costs PER MONTH
7. "revenue_streams": ONLY the streams that follow from the business
   model in rule 5. Do NOT add advertising, subscriptions, or extra
   services unless the idea states them. One stream is fine.
   Maximum 3 financial_risks and 3 recommendations.
   Every recommendation must be about costs, pricing, or unit economics.
   Do NOT recommend technology, algorithms, or product features.
   Keep every item to one short sentence.
8. Do not invent competitors, statistics, or facts.
9. Write text in clear, simple Modern Standard Arabic.

Return ONLY valid JSON with exactly these keys:

{
  "business_model": "",
  "revenue_streams": [],
  "assumptions": {
    "average_price": 0,
    "commission_rate": 0,
    "variable_cost_per_order": 0,
    "fixed_costs": 0
  },
  "financial_risks": [],
  "recommendations": []
}
"""

MAHMOUD_KEYS = [
    "business_model", "revenue_streams", "assumptions",
    "financial_risks", "recommendations"
]


def mahmoud_finance_analysis(idea, understanding):
    messages = [
        {"role": "system", "content": MAHMOUD_PROMPT},
        {
            "role": "user",
            "content": (
                f"BUSINESS IDEA:\n{idea}\n\n"
                f"UNDERSTANDING:\n{json.dumps(understanding, ensure_ascii=False)}"
            )
        }
    ]

    data = run_json_agent("Mahmoud", messages, 450, MAHMOUD_KEYS)

    return {
        "business_model": str(data["business_model"]).strip(),
        "revenue_streams": clean_text_list(data["revenue_streams"], "revenue_streams"),
        "assumptions": validate_assumptions(data["assumptions"]),
        "financial_risks": clean_text_list(data["financial_risks"], "financial_risks"),
        "recommendations": clean_text_list(data["recommendations"], "recommendations"),
        "assumptions_source": "llm_estimate",
        "assumptions_note": "الأرقام تقديرية من النموذج وليست من المستخدم أو من المصادر، وكل التكاليف بالجنيه المصري والتكاليف الثابتة شهرية."
    }

In [ ]:
def to_score(value, name):
    # A number from 1 to 5 (used by Sherif and Manal)
    number = to_number(value, name)

    if not 1 <= number <= 5:
        raise ValueError(f"{name} must be between 1 and 5, got: {value!r}")

    return int(round(number))


def clean_execution_risks(items):
    if not isinstance(items, list):
        raise ValueError("Sherif: execution_risks must be a list")

    risks = []
    for item in items[:5]:
        if not isinstance(item, dict):
            raise ValueError("Sherif: each execution risk must be an object")

        text = str(item.get("risk", "")).strip()
        if not text:
            continue

        risks.append({
            "risk": text,
            "severity": to_score(item.get("severity"), "Sherif severity")
        })

    if not risks:
        raise ValueError("Sherif: no execution risks returned")

    return risks


SHERIF_PROMPT = """
You are Sherif, the Operations & Execution Expert
for an Egyptian startup evaluation platform.

Decide HOW the idea can be executed, starting from the
smallest realistic MVP that can test the core hypothesis.

Rules:
1. Validation before scale. The MVP must be the smallest version that
   tests the core business hypothesis. Things that can be done manually
   must NOT be in must_have.
2. Do NOT put a full app, payment systems, delivery tracking, AI features,
   or complex infrastructure in must_have unless the idea cannot be
   tested without them.
3. Do NOT name specific companies, products, payment providers,
   delivery companies, prices, or legal requirements.
   Describe the capability needed instead.
4. Do NOT recommend fundraising as an early step.
5. Be practical and skeptical. Do not claim that demand or profit exists.
6. "mvp_complexity": a number from 1 to 5 for the effort to build and run
   the MVP (1 = easy to run manually, 5 = very complex).
7. Each execution risk has "severity", a number from 1 to 5
   (1 = minor, 5 = could stop the business).
8. Maximum 5 must_have, 3 main_bottlenecks, 4 success_metrics,
   5 execution_risks. Keep every item to one short sentence.
9. "verdict": one neutral sentence about how to start.
   Do NOT make claims about demand, profit, or the market.
10. Write text in clear, simple Modern Standard Arabic.

Return ONLY valid JSON with exactly these keys:

{
  "core_hypothesis": "",
  "mvp": {
    "description": "",
    "must_have": []
  },
  "mvp_complexity": 3,
  "main_bottlenecks": [],
  "success_metrics": [],
  "execution_risks": [
    {"risk": "", "severity": 3}
  ],
  "verdict": ""
}
"""

SHERIF_KEYS = [
    "core_hypothesis", "mvp", "mvp_complexity", "main_bottlenecks",
    "success_metrics", "execution_risks", "verdict"
]


def sherif_execution_analysis(idea, understanding):
    messages = [
        {"role": "system", "content": SHERIF_PROMPT},
        {
            "role": "user",
            "content": (
                f"BUSINESS IDEA:\n{idea}\n\n"
                f"UNDERSTANDING:\n{json.dumps(understanding, ensure_ascii=False)}"
            )
        }
    ]

    data = run_json_agent("Sherif", messages, 800, SHERIF_KEYS)

    if not isinstance(data["mvp"], dict) or "description" not in data["mvp"] or "must_have" not in data["mvp"]:
        raise ValueError("Sherif: mvp must contain description and must_have")

    return {
        "core_hypothesis": str(data["core_hypothesis"]).strip(),
        "mvp": {
            "description": str(data["mvp"]["description"]).strip(),
            "must_have": clean_text_list(data["mvp"]["must_have"], "must_have", limit=5)
        },
        "mvp_complexity": to_score(data["mvp_complexity"], "Sherif mvp_complexity"),
        "main_bottlenecks": clean_text_list(data["main_bottlenecks"], "main_bottlenecks", limit=3),
        "success_metrics": clean_text_list(data["success_metrics"], "success_metrics", limit=4),
        "execution_risks": clean_execution_risks(data["execution_risks"]),
        "verdict": str(data["verdict"]).strip()
    }

In [ ]:
def clean_critical_risks(items):
    if not isinstance(items, list):
        raise ValueError("Manal: critical_risks must be a list")

    risks = []
    for item in items[:5]:
        if not isinstance(item, dict):
            raise ValueError("Manal: each risk must be an object")

        text = str(item.get("risk", "")).strip()
        if not text:
            continue

        risks.append({
            "risk": text,
            "likelihood": to_score(item.get("likelihood"), "Manal likelihood"),
            "impact": to_score(item.get("impact"), "Manal impact"),
            "why_it_matters": str(item.get("why_it_matters", "")).strip(),
            "mitigation": str(item.get("mitigation", "")).strip()
        })

    if not risks:
        raise ValueError("Manal: no risks returned")

    return risks


MANAL_PROMPT = """
You are Manal, the Risk & Strategy Analyst for an Egyptian startup
evaluation platform. You are the Devil's Advocate.

Find the assumptions and risks that could make this business fail.

Rules:
1. Never invent market data, statistics, regulations, competitors,
   companies, prices, or legal requirements.
2. Never invent sample sizes or numbers (like "50 users").
   If a test needs participants, say "a small pilot".
3. Do not make legal claims. If legal checking is needed, say that
   local professional verification is required.
4. Use only the business idea and its understanding.
   Do not rely on other analysts.
5. Challenge the assumptions behind the idea instead of repeating it.
   Focus on demand, trust, quality control, pricing, willingness to pay,
   and regulation that needs verification, not only on logistics.
6. Do not claim that demand or profit exists.
   "strongest_reason_to_succeed" must be conditional
   (it can succeed IF something is true).
7. Each risk has "likelihood" and "impact", each a number from 1 to 5
   (1 = very low, 5 = very high). Do not use percentages.
8. Maximum 5 critical_risks, 4 core_assumptions, 4 red_flags.
   Keep every text to one short sentence.
9. Write text in clear, simple Modern Standard Arabic.

Return ONLY valid JSON with exactly these keys:

{
  "core_assumptions": [],
  "critical_risks": [
    {
      "risk": "",
      "likelihood": 3,
      "impact": 3,
      "why_it_matters": "",
      "mitigation": ""
    }
  ],
  "strongest_reason_to_fail": "",
  "strongest_reason_to_succeed": "",
  "red_flags": [],
  "strategic_verdict": ""
}
"""

MANAL_KEYS = [
    "core_assumptions", "critical_risks", "strongest_reason_to_fail",
    "strongest_reason_to_succeed", "red_flags", "strategic_verdict"
]


def manal_risk_analysis(idea, understanding):
    messages = [
        {"role": "system", "content": MANAL_PROMPT},
        {
            "role": "user",
            "content": (
                f"BUSINESS IDEA:\n{idea}\n\n"
                f"UNDERSTANDING:\n{json.dumps(understanding, ensure_ascii=False)}"
            )
        }
    ]

    data = run_json_agent("Manal", messages, 800, MANAL_KEYS)

    return {
        "core_assumptions": clean_text_list(data["core_assumptions"], "core_assumptions", limit=4),
        "critical_risks": clean_critical_risks(data["critical_risks"]),
        "strongest_reason_to_fail": str(data["strongest_reason_to_fail"]).strip(),
        "strongest_reason_to_succeed": str(data["strongest_reason_to_succeed"]).strip(),
        "red_flags": clean_text_list(data["red_flags"], "red_flags", limit=4),
        "strategic_verdict": str(data["strategic_verdict"]).strip()
    }

In [ ]:
import math


def calculate_financials(assumptions):
    price = assumptions["average_price"]
    commission = assumptions["commission_rate"]
    variable_cost = assumptions["variable_cost_per_order"]
    fixed_costs = assumptions["fixed_costs"]

    # Revenue the platform keeps from one order
    revenue_per_order = round(price * commission, 2)
    contribution_per_order = round(revenue_per_order - variable_cost, 2)
    contribution_margin = round(contribution_per_order / revenue_per_order, 2)

    # Lowest commission that makes one order profitable
    min_commission_rate = round(variable_cost / price, 4)

    if contribution_per_order > 0:
        break_even_orders = math.ceil(fixed_costs / contribution_per_order)
        status = "Profitable per order"
    else:
        break_even_orders = None
        status = "Not achievable under current assumptions"

    return {
        "revenue_per_order": revenue_per_order,
        "contribution_per_order": contribution_per_order,
        "contribution_margin": contribution_margin,
        "break_even_orders_per_month": break_even_orders,
        "min_commission_rate": min_commission_rate,
        "status": status
    }

In [ ]:
ANALYSTS = ["Mona", "Mahmoud", "Sherif", "Manal"]
ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")


def build_board_context(idea, understanding, mona, mahmoud, financials, sherif, manal):
    assumptions = mahmoud["assumptions"]

    return {
        "idea": idea.strip(),
        "missing_information": understanding["missing_information"],
        "Mona": {
            "market_overview": mona["market_overview"],
            "direct_competitors": [{"name": c["name"], "reason": c["reason"]} for c in mona["direct_competitors"]],
            "indirect_competitors": [{"name": c["name"], "reason": c["reason"]} for c in mona["indirect_competitors"]],
            "similar_global_businesses": [{"name": c["name"], "reason": c["reason"]} for c in mona["similar_global_businesses"]],
            "market_gaps": [x["text"] for x in mona["market_gaps"]],
            "market_risks": [x["text"] for x in mona["market_risks"]],
            "verdict": mona["verdict"]
        },
        "Mahmoud": {
            "numbers_are_estimates": True,
            "business_model": mahmoud["business_model"],
            "average_price_egp": assumptions["average_price"],
            "commission_rate_percent": round(assumptions["commission_rate"] * 100, 1),
            "variable_cost_per_order_egp": assumptions["variable_cost_per_order"],
            "fixed_costs_per_month_egp": assumptions["fixed_costs"],
            "revenue_per_order_egp": financials["revenue_per_order"],
            "contribution_per_order_egp": financials["contribution_per_order"],
            "contribution_margin_percent": round(financials["contribution_margin"] * 100, 1),
            "break_even_orders_per_month": financials["break_even_orders_per_month"],
            "min_commission_rate_percent": round(financials["min_commission_rate"] * 100, 1),
            "financial_status": financials["status"],
            "financial_risks": mahmoud["financial_risks"]
        },
        "Sherif": {
            "core_hypothesis": sherif["core_hypothesis"],
            "mvp_description": sherif["mvp"]["description"],
            "mvp_complexity_1_to_5": sherif["mvp_complexity"],
            "main_bottlenecks": sherif["main_bottlenecks"],
            "execution_risks": sherif["execution_risks"]
        },
        "Manal": {
            "core_assumptions": manal["core_assumptions"],
            "critical_risks": [
                {"risk": r["risk"], "likelihood": r["likelihood"], "impact": r["impact"]}
                for r in manal["critical_risks"]
            ],
            "strongest_reason_to_fail": manal["strongest_reason_to_fail"],
            "strongest_reason_to_succeed": manal["strongest_reason_to_succeed"],
            "red_flags": manal["red_flags"]
        }
    }


def numbers_in(text):
    text = str(text).translate(ARABIC_DIGITS).replace(",", "")
    return {format(float(n), "g") for n in re.findall(r"\d+(?:\.\d+)?", text)}


def as_short_list(value, name, limit=3):
    if not isinstance(value, list):
        raise ValueError(f"Board: {name} must be a list")
    return [str(x).strip() for x in value if str(x).strip()][:limit]


BOARD_PROMPT = """
You are Ramadan, the moderator of an Egyptian startup "Shark Board".

Four analysts evaluate one business idea:
Mona = Market & Competition
Mahmoud = Finance
Sherif = Operations & Execution
Manal = Risk & Strategy

Write a SHORT discussion using ONLY the BOARD DATA.

Rules:
1. Never add outside facts, statistics, or companies.
   Mention only competitors that are listed in BOARD DATA.
2. Copy numbers exactly as they appear. Never calculate or change them.
3. All financial numbers are ESTIMATES made by the program from
   assumptions. Mahmoud must say they are estimates.
   If contribution_per_order_egp is 0 or negative, Mahmoud must say:
   under the estimated assumptions the profit per order is not positive,
   and the commission would have to be ABOVE min_commission_rate_percent
   for the profit per order to become positive.
   Never say the business model or the idea is unachievable or failing,
   and never say profitability is impossible.
4. break_even_orders_per_month = null means break-even cannot be reached
   with these assumptions.
5. Never say that demand or profit exists unless BOARD DATA says so.
6. If Mona found no direct competitor, say that the sources did not
   prove one, not that there is no competition.
7. Each analyst speaks only within their own role, 1-2 short sentences.
8. You (Ramadan) are neutral. Do NOT give a final decision, a score,
   or invest/pass advice.
9. Together the discussion must cover: the strongest market point,
   the biggest financial problem, the biggest execution challenge,
   the biggest strategic risk.
10. key_disagreements: ONLY real disagreements between two analysts,
    where each item names both analysts and the point they differ on.
    If there is no real disagreement, return [].
    Do NOT write category labels or prefixes.
11. board_conclusion: 1-2 neutral sentences about what must be validated
    before scaling.
12. Maximum 3 items in each list. Keep every item to one short sentence.
13. Write everything in Arabic only. English is allowed ONLY for company
    names exactly as written in BOARD DATA. Never mix English letters
    inside an Arabic word.

Return ONLY valid JSON with exactly these keys:

{
  "discussion": [
    {"speaker": "Mona", "message": ""},
    {"speaker": "Mahmoud", "message": ""},
    {"speaker": "Sherif", "message": ""},
    {"speaker": "Manal", "message": ""}
  ],
  "key_disagreements": [],
  "shared_concerns": [],
  "critical_questions": [],
  "board_conclusion": ""
}
"""

BOARD_KEYS = [
    "discussion", "key_disagreements", "shared_concerns",
    "critical_questions", "board_conclusion"
]

def board_text(result):
    parts = [t["message"] for t in result["discussion"]]
    for key in ["key_disagreements", "shared_concerns", "critical_questions"]:
        parts += result[key]
    parts.append(result["board_conclusion"])
    return " ".join(parts)


def board_warnings(result, board_data, context):
    warnings = []
    text = board_text(result)

    # Numbers that are not in BOARD DATA
    for n in sorted(numbers_in(text) - numbers_in(context)):
        warnings.append(f"number not in board data: {n}")

    # English words that are not names from BOARD DATA
    names = {w.lower() for w in ANALYSTS}
    for group in ["direct_competitors", "indirect_competitors", "similar_global_businesses"]:
        for c in board_data["Mona"][group]:
            names.update(re.findall(r"[A-Za-z]+", c["name"].lower()))

    for word in sorted(set(re.findall(r"[A-Za-z]+", text))):
        if word.lower() not in names:
            warnings.append(f"unknown English word: {word}")

    # Mahmoud must label the numbers as estimates
    mahmoud_message = next(t["message"] for t in result["discussion"] if t["speaker"] == "Mahmoud")
    if "تقدير" not in mahmoud_message and "افتراض" not in mahmoud_message:
        warnings.append("Mahmoud did not label the numbers as estimates")

    return warnings
def board_discussion(board_data):
    context = json.dumps(board_data, ensure_ascii=False, indent=2)

    messages = [
        {"role": "system", "content": BOARD_PROMPT},
        {"role": "user", "content": f"BOARD DATA:\n{context}"}
    ]

    data = run_json_agent("Board", messages, 1000, BOARD_KEYS)

    # Each analyst must speak exactly once
    if not isinstance(data["discussion"], list):
        raise ValueError("Board: discussion must be a list")

    spoken = {}
    for turn in data["discussion"]:
        if not isinstance(turn, dict):
            raise ValueError("Board: each discussion turn must be an object")

        speaker = str(turn.get("speaker", "")).strip()
        message = str(turn.get("message", "")).strip()

        if speaker not in ANALYSTS or speaker in spoken or not message:
            raise ValueError(f"Board: invalid or duplicate speaker/message: {speaker!r}")

        spoken[speaker] = message

    if set(spoken) != set(ANALYSTS):
        raise ValueError(f"Board: missing speakers: {sorted(set(ANALYSTS) - set(spoken))}")

    result = {
        "moderator": "Ramadan",
        "discussion": [{"speaker": s, "message": spoken[s]} for s in ANALYSTS],
        "key_disagreements": as_short_list(data["key_disagreements"], "key_disagreements"),
        "shared_concerns": as_short_list(data["shared_concerns"], "shared_concerns"),
        "critical_questions": as_short_list(data["critical_questions"], "critical_questions"),
        "board_conclusion": str(data["board_conclusion"]).strip()
    }

    result["warnings"] = board_warnings(result, board_data, context)

    return result

In [ ]:
SCORE_WEIGHTS = {
    "Market Opportunity": 0.20,
    "Financial Feasibility": 0.15,
    "Execution Feasibility": 0.20,
    "Competition": 0.15,
    "Scalability": 0.15,
    "Risk": 0.15
}


def clamp(value, low=0.0, high=10.0):
    return max(low, min(high, value))


def score_market(mona):
    score = 5.0
    notes = ["base 5 (no direct demand data)"]

    if mona["direct_competitors"]:
        score += 1.5
        notes.append("+1.5 a direct competitor is proven in Egypt (the need is already served)")
    if mona["similar_global_businesses"]:
        score += 1
        notes.append("+1 a similar model is proven abroad")
    if mona["market_gaps"]:
        score += 1
        notes.append("+1 a market gap is supported by a source quote")
    if mona["market_risks"]:
        score -= 1
        notes.append("-1 a market risk is supported by a source quote")

    return clamp(score), "; ".join(notes)


def score_financial(financials):
    margin = financials["contribution_margin"]
    orders = financials["break_even_orders_per_month"]

    if margin <= 0:
        score = clamp(2 * (1 + margin / 0.5), 0, 2)
        notes = [f"margin {margin:.0%}: profit per order is not positive (0-2)"]
    else:
        score = 2 + min(margin, 0.6) / 0.6 * 6
        notes = [f"margin {margin:.0%}: positive profit per order (2-8)"]

    if orders is None:
        notes.append("no break-even under the assumptions")
    elif orders <= 500:
        score += 2
        notes.append(f"+2 break-even at {orders} orders per month")
    elif orders <= 2000:
        score += 1
        notes.append(f"+1 break-even at {orders} orders per month")
    else:
        notes.append(f"break-even needs {orders} orders per month")

    return clamp(score), "; ".join(notes)


def score_execution(sherif):
    complexity = sherif["mvp_complexity"]
    severities = [r["severity"] for r in sherif["execution_risks"]]
    average_severity = sum(severities) / len(severities)

    complexity_score = (5 - complexity) / 4 * 10
    severity_score = (5 - average_severity) / 4 * 10
    score = 0.4 * complexity_score + 0.6 * severity_score

    note = f"MVP complexity {complexity}/5 (40%), average execution risk severity {average_severity:.1f}/5 (60%)"
    return clamp(score), note


def score_competition(mona):
    if mona["direct_competitors"]:
        return 3.0, "a direct competitor is proven to operate in Egypt"
    if mona["indirect_competitors"]:
        return 6.0, "no direct competitor proven, only indirect ones"
    return 5.0, "no competitor proven by the sources (uncertain, not the same as no competition)"


def score_scalability(financials, sherif):
    margin = financials["contribution_margin"]
    worst_severity = max(r["severity"] for r in sherif["execution_risks"])

    unit_score = min(max(margin, 0) / 0.5, 1) * 10
    bottleneck_score = (5 - worst_severity) / 4 * 10
    score = 0.6 * unit_score + 0.4 * bottleneck_score

    note = f"margin {margin:.0%} (60%), worst execution risk severity {worst_severity}/5 (40%)"
    return clamp(score), note


def score_risk(manal):
    products = [r["likelihood"] * r["impact"] for r in manal["critical_risks"]]
    level = 0.5 * max(products) + 0.5 * (sum(products) / len(products))

    # level goes from 1 (safe) to 25 (very risky)
    score = 10 * (1 - (level - 1) / 24)
    note = f"worst risk {max(products)}/25, average risk {sum(products) / len(products):.1f}/25 (likelihood x impact)"
    return clamp(score), note


def calculate_final_score(mona, financials, sherif, manal):
    results = {
        "Market Opportunity": score_market(mona),
        "Financial Feasibility": score_financial(financials),
        "Execution Feasibility": score_execution(sherif),
        "Competition": score_competition(mona),
        "Scalability": score_scalability(financials, sherif),
        "Risk": score_risk(manal)
    }

    final_score = sum(results[name][0] * SCORE_WEIGHTS[name] for name in results) * 10

    return {
        "subscores": {name: round(results[name][0], 1) for name in results},
        "weights": SCORE_WEIGHTS,
        "final_score": round(final_score, 1),
        "explanations": {name: results[name][1] for name in results},
        "caveats": ["الدرجات المالية مبنية على افتراضات تقديرية من النموذج وليست من المستخدم"]
    }

In [ ]:
INVEST_SCORE = 70
CONDITIONS_SCORE = 50
MIN_SUBSCORE_FOR_INVEST = 4

DIMENSIONS_AR = {
    "Market Opportunity": "فرصة السوق",
    "Financial Feasibility": "الجدوى المالية",
    "Execution Feasibility": "جدوى التنفيذ",
    "Competition": "المنافسة",
    "Scalability": "قابلية التوسع",
    "Risk": "المخاطر"
}


def decide(final_score, subscores):
    if final_score >= INVEST_SCORE and min(subscores.values()) >= MIN_SUBSCORE_FOR_INVEST:
        return "INVEST"
    if final_score >= CONDITIONS_SCORE:
        return "INVEST WITH CONDITIONS"
    return "PASS"


def competitor_names(mona):
    items = mona["direct_competitors"] or mona["indirect_competitors"]
    return [c["name"] for c in items]


def weakness_text(name, mona, financials, sherif, manal):
    top_risk = max(manal["critical_risks"], key=lambda r: r["likelihood"] * r["impact"])
    worst_execution = max(sherif["execution_risks"], key=lambda r: r["severity"])

    if name == "Financial Feasibility":
        if financials["contribution_per_order"] <= 0:
            rate = round(financials["min_commission_rate"] * 100, 1)
            return (f"الربح لكل طلب {financials['contribution_per_order']} جنيه تحت الافتراضات التقديرية، "
                    f"والعمولة يجب أن تزيد عن {rate}% ليصبح الربح موجبًا")
        return f"نقطة التعادل {financials['break_even_orders_per_month']} طلب في الشهر تحت الافتراضات التقديرية"

    if name == "Risk":
        return f"أكبر خطر: {top_risk['risk']} (احتمال {top_risk['likelihood']} × أثر {top_risk['impact']})"

    if name == "Execution Feasibility":
        return f"أخطر عائق تنفيذي: {worst_execution['risk']} (الشدة {worst_execution['severity']}/5)"

    if name == "Scalability":
        return f"هامش الربح {financials['contribution_margin']:.0%} وأخطر عائق تنفيذي: {worst_execution['risk']}"

    if name == "Competition":
        names = competitor_names(mona)
        if names:
            return "منافسون مثبتون في المصادر: " + "، ".join(names)
        return "لم تثبت المصادر منافسين، ولا يعني ذلك غياب المنافسة"

    # Market Opportunity
    if mona["market_risks"]:
        return mona["market_risks"][0]["text"]
    return "لا توجد في المصادر بيانات طلب مباشرة على الفكرة"


def action_text(name, mona, financials, sherif, manal):
    top_risk = max(manal["critical_risks"], key=lambda r: r["likelihood"] * r["impact"])

    if name == "Financial Feasibility":
        rate = round(financials["min_commission_rate"] * 100, 1)
        return (f"اختبار افتراضات السعر والعمولة والتكلفة بأرقام حقيقية من تجربة صغيرة، "
                f"والهدف عمولة أعلى من {rate}% مع بقاء العرض جذابًا للطرفين")

    if name == "Risk":
        step = top_risk["mitigation"] or f"وضع خطة تخفيف لأكبر خطر: {top_risk['risk']}"
        return f"تنفيذ إجراء تخفيف لأكبر خطر: {step}"

    if name == "Execution Feasibility":
        metrics = sherif["success_metrics"]
        metric = f"، وقياس: {metrics[0]}" if metrics else ""
        return f"تشغيل نسخة MVP يدوية وبسيطة: {sherif['mvp']['description']}{metric}"

    if name == "Scalability":
        bottlenecks = sherif["main_bottlenecks"]
        if bottlenecks:
            return f"إيجاد حل لأكبر عائق تشغيلي يمنع التوسع: {bottlenecks[0]}"
        return "تحديد أكبر عائق تشغيلي يمنع التوسع وإيجاد حل له"

    if name == "Competition":
        names = competitor_names(mona)
        if names:
            return f"مقارنة العرض بعرض {names[0]} وتحديد نقطة تميّز واضحة قابلة للاختبار"
        return "بحث إضافي لإثبات المنافسين الفعليين في السوق"

    # Market Opportunity
    return "اختبار الطلب الفعلي عبر تجربة صغيرة مع عملاء حقيقيين قبل أي بناء"


def final_judge(score_result, mona, financials, sherif, manal):
    subscores = score_result["subscores"]
    final_score = score_result["final_score"]
    decision = decide(final_score, subscores)

    # Weakest first
    ranked = sorted(subscores, key=lambda n: subscores[n])

    reasons = {
        "INVEST": "الدرجة النهائية مرتفعة ولا يوجد محور ضعيف",
        "INVEST WITH CONDITIONS": "الدرجة النهائية متوسطة، والفكرة تحتاج معالجة نقاط الضعف قبل الاستثمار",
        "PASS": "الدرجة النهائية منخفضة تحت الافتراضات الحالية، ولا يوصى بالاستثمار قبل معالجة نقاط الضعف"
    }

    weaknesses = [
        {
            "dimension": DIMENSIONS_AR[name],
            "score": subscores[name],
            "detail": weakness_text(name, mona, financials, sherif, manal)
        }
        for name in ranked if subscores[name] < 5
    ]

    strengths = [
        {
            "dimension": DIMENSIONS_AR[name],
            "score": subscores[name],
            "detail": score_result["explanations"][name]
        }
        for name in reversed(ranked) if subscores[name] >= 6
    ]

    # One week for each of the 3 weakest dimensions, then a review week
    plan = [
        {
            "week": week,
            "focus": DIMENSIONS_AR[name],
            "action": action_text(name, mona, financials, sherif, manal)
        }
        for week, name in enumerate(ranked[:3], 1)
    ]
    plan.append({
        "week": 4,
        "focus": "المراجعة",
        "action": "إعادة تشغيل التحليل بالأرقام الحقيقية التي جُمعت في الأسابيع السابقة واتخاذ قرار الاستمرار أو التعديل أو التوقف"
    })

    return {
        "decision": decision,
        "final_score": final_score,
        "decision_reason": f"{reasons[decision]} ({final_score}/100)",
        "decision_rule": (
            f"INVEST: الدرجة ≥ {INVEST_SCORE} وكل المحاور ≥ {MIN_SUBSCORE_FOR_INVEST} | "
            f"INVEST WITH CONDITIONS: الدرجة ≥ {CONDITIONS_SCORE} | PASS: أقل من ذلك"
        ),
        "strengths": strengths,
        "weaknesses": weaknesses,
        "action_plan_30_days": plan,
        "caveats": score_result["caveats"]
    }

In [ ]:
import difflib

PERSONAS = {
    "Mona": {
        "name": "أستاذة منى",
        "role": "Market & Competition",
        "focus": "who the customer is, what makes the idea different from similar services, and how customers will find it",
        "character": "A sharp, calm, well-organized market researcher. Polite but direct.",
        # Do not put ready-made questions in the generation prompt: Qwen may copy them.
        # The role, focus and character below are sufficient to define Mona's voice.
        "examples": [
            "هذا فرق مهم، لكنه ما زال يحتاج إلى دليل على أن العميل يهتم به.",
            "الصورة أوضح الآن، لكن لا أرى دليلًا على أن الناس ستغيّر عادتها."
        ]
    },
    "Mahmoud": {
        "name": "الحاج عمران",
        "role": "Finance",
        "focus": "how the business earns money (who pays the platform and how much) and its main costs",
        "character": "An old-school merchant with 40 years in the market. Practical, patient and funny, trusts only clear numbers. Always addresses the founder as 'يا بشمهندس' or 'يا ابني'.",
        "examples": [
            "يا بشمهندس، الحساب لا يكذب. من أين ستأتي أرباحك بالضبط؟",
            "كلام جميل يا ابني، لكن كم ستكلّفك كل عملية؟"
        ]
    },
    "Sherif": {
        "name": "مستر شريف",
        "role": "Operations & Execution",
        "focus": "how the first version will actually run in the first month, who does what, and what will be manual at the start",
        "character": "A fast, practical operations engineer who loves small concrete steps. Naturally uses the English words MVP, pilot, workflow and bottleneck inside his Arabic.",
        "examples": [
            "حسنًا، لكن ماذا ستفعل في اليوم الأول بالضبط؟",
            "هل يمكن أن يعمل الـMVP يدويًا في البداية؟"
        ]
    },
    "Manal": {
        "name": "مدام منال",
        "role": "Risk & Strategy",
        "focus": "the biggest assumption that could make the idea fail, and what the founder will do if it does",
        "character": "A strict, sharp-tongued risk consultant with calm, dry sarcasm. The devil's advocate. Never flatters, never rude.",
        "examples": [
            "جميل. وماذا بعد؟ لو فشل هذا الافتراض فماذا ستفعل؟",
            "أنا لا أجامل، أنا أبحث عن الثغرة. ما الذي فاتك؟"
        ]
    }
}

# Examples for reactions: statements only, no questions
REACT_EXAMPLES = {
    "Mona": [
        "هذا فرق مهم، لكنه ما زال يحتاج إلى دليل على أن العميل يهتم به.",
        "الصورة أوضح الآن، لكن لا أرى دليلًا على أن الناس ستغيّر عادتها."
    ],
    "Mahmoud": [
        "عمولة فقط؟ حسنًا يا بشمهندس، لكن بدون نسبة واضحة لن يكتمل الحساب.",
        "هذا رقم جيد يا ابني، لكن الحساب ما زال ناقصًا."
    ],
    "Sherif": [
        "هذه بداية عملية، لكن يجب كتابة الـworkflow خطوة بخطوة.",
        "جيد أن تبدأ يدويًا، لكن الـbottleneck ما زال غير واضح."
    ],
    "Manal": [
        "هذا رد مريح، لكنه ليس دليلًا، والافتراض ما زال يحتاج إلى اختبار.",
        "كلام جميل، لكني ما زلت أرى ثغرة في الخطة."
    ]
}

# Used by the NileChat cell to reject formal Arabic words
MSA_WORDS = {"ما", "ماذا", "كيف", "لماذا", "لكن", "هل", "الذي", "التي", "سوف", "ليس", "أين", "متى"}

ASK_TASK = """Ask the founder exactly ONE short question (a single question, not two questions joined by "and") about: {focus}.
Do not ask about anything the founder already said."""

REACT_TASK = """React to the founder's answer to YOUR question in 1-2 short sentences,
from your own point of view: say what is clear, or point out the gap.
Do NOT ask any question. Do NOT give a final decision, a score, or invest/pass advice."""


def persona_prompt(key, mode):
    p = PERSONAS[key]
    task = ASK_TASK.format(focus=p["focus"]) if mode == "ask" else REACT_TASK
    shown = p["examples"] if mode == "ask" else REACT_EXAMPLES[key]
    examples = "\n".join(f"- {x}" for x in shown)

    return f"""
You are {p['name']}, a member of the "Jow El Tank" startup panel,
talking directly to the founder in a live conversation.

Your role: {p['role']}
Your character: {p['character']}

Write in clear, simple Modern Standard Arabic (another model will turn it into Egyptian dialect later).
Keep your character: the way you address the founder and the English terms your character naturally uses.

Examples of how you talk (voice only, never reuse their words):
{examples}

Rules:
1. Talk to the founder directly, in your own character.
2. Use ONLY the idea, the understanding and what the founder said.
   Never invent facts, statistics, prices, competitors or companies.
   Mention a number only if the founder said it.
3. Treat what the founder says as his claims, not as verified facts.
4. Maximum 2 short sentences. Never copy a line from the conversation.
5. Return only your spoken words: no name prefix, no quotes, no markdown.

{task}
"""


def build_context(key, mode, transcript):
    founder_said = [t["text"] for t in transcript if t["speaker"] == "Founder"]
    said = "\n".join(f"- {x}" for x in founder_said) or "(nothing yet)"

    text = f"WHAT THE FOUNDER HAS SAID SO FAR:\n{said}"

    if mode == "react":
        my_questions = [t["text"] for t in transcript if t["speaker"] == key]
        if not my_questions or not founder_said:
            raise ValueError("react needs your own question and the founder's answer in the transcript")
        text += f"\n\nYOUR QUESTION: {my_questions[-1]}\nTHE FOUNDER'S ANSWER: {founder_said[-1]}"

    return text


def clean_reply(text, name):
    text = text.strip().strip('"').strip().replace("?", "؟")

    # Remove the character name if the model wrote it
    if text.startswith(name):
        text = text[len(name):].lstrip(" :،-")

    text = " ".join(text.split())

    # If the reply was cut in the middle of a sentence, keep the complete part
    last_end = max(text.rfind(c) for c in ".؟!")
    if last_end != -1 and last_end < len(text) - 1:
        text = text[:last_end + 1]

    return text


def drop_questions(text):
    # Keep only the sentences that are not questions
    sentences = re.findall(r"[^.؟!]+[.؟!]?", text)
    kept = [s.strip() for s in sentences if "؟" not in s and s.strip()]
    return " ".join(kept)


def msa_problem(key, reply, mode, transcript, context_text):
    if not reply:
        return "empty reply"
    if len(reply) > 220:
        return "too long"
    if mode == "ask" and (reply.count("؟") != 1 or not reply.endswith("؟")):
        return "an ask must be exactly one short question"
    if mode == "react" and "؟" in reply:
        return "a reaction must not ask a question"
    if numbers_in(reply) - numbers_in(context_text):
        return "it uses a number nobody said"

    allowed = {w.lower() for w in re.findall(r"[A-Za-z]+", context_text)} | ALLOWED_ENGLISH
    extra = {w for w in re.findall(r"[A-Za-z]+", reply) if w.lower() not in allowed}
    if extra:
        return f"English words that are not allowed: {sorted(extra)}"

    for turn in transcript:
        old = turn["text"]
        if len(old) > 15 and (old in reply or reply in old):
            return "repeats an earlier line"

    for example in PERSONAS[key]["examples"] + REACT_EXAMPLES[key]:
        if difflib.SequenceMatcher(None, reply, example).ratio() > 0.8:
            return "copied an example"

    return None


def persona_say(key, mode, idea, understanding, transcript, tries=4):
    p = PERSONAS[key]
    now = "Now ask your question." if mode == "ask" else "Now react to the founder's answer."
    understanding_text = json.dumps(understanding, ensure_ascii=False)

    messages = [
        {"role": "system", "content": persona_prompt(key, mode)},
        {
            "role": "user",
            "content": (
                f"BUSINESS IDEA:\n{idea}\n\n"
                f"UNDERSTANDING:\n{understanding_text}\n\n"
                f"{build_context(key, mode, transcript)}\n\n{now}"
            )
        }
    ]

    # Everything the persona is allowed to mention (used to check numbers and English words)
    context_text = idea + " " + understanding_text + " " + " ".join(
        t["text"] for t in transcript if t["speaker"] == "Founder"
    )

    for _ in range(tries):
        reply = clean_reply(qwen_generate(messages, max_new_tokens=120, temperature=0.5), p["name"])

        # A reaction must not ask: remove the question sentences
        if mode == "react":
            reply = drop_questions(reply)

        problem = msa_problem(key, reply, mode, transcript, context_text)

        if problem is None:
            return to_egyptian(key, reply)

        # Tell the model what was wrong, then try again
        messages = messages + [
            {"role": "assistant", "content": reply or "(empty)"},
            {"role": "user", "content": f"Rejected: {problem}. Write a new reply that fixes this."}
        ]

    raise ValueError(f"{p['name']} failed after {tries} tries: {problem}. Last reply: {reply}")

In [ ]:
# Examples for reactions: statements only, no questions
REACT_EXAMPLES = {
    "Mona": [
        "الفرق ده كويس، بس لسه محتاج يتثبت إن العميل هيهتم بيه فعلًا.",
        "كده الصورة بقت أوضح شوية، بس مفيش دليل لسه إن الناس هتغيّر عادتها."
    ],
    "Mahmoud": [
        "عمولة وخلاص؟ ماشي يا بشمهندس، بس من غير نسبة واضحة الحساب مش هيجمع.",
        "الرقم ده بداية كويسة يا ابني، بس الحساب لسه مش كامل."
    ],
    "Sherif": [
        "ده بداية عملية، بس لازم الـworkflow يتكتب خطوة بخطوة.",
        "كويس إنك هتبدأ يدوي، بس الـbottleneck لسه مش واضح."
    ],
    "Manal": [
        "ده رد مريح، بس مش دليل. الافتراض ده لسه محتاج اختبار.",
        "كلام حلو، بس أنا لسه شايفة ثغرة في الخطة."
    ]
}

In [ ]:
NILE_MODEL_NAME = "UBC-NLP/NileChat-3B"

tokenizer_nile = AutoTokenizer.from_pretrained(NILE_MODEL_NAME, token=True)

model_nile = AutoModelForCausalLM.from_pretrained(
    NILE_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    token=True
)

used = sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / 1024**3
print(f"NileChat loaded ✅  (VRAM in use: {used:.1f} GB)")

In [ ]:
import difflib

ALLOWED_ENGLISH = {"mvp", "pilot", "workflow", "bottleneck"}
# Arabic LETTERS only. The range \u0600-\u06FF also contains the Arabic question mark and comma,
# which used to stick to the word before them and break the word comparisons.
ARABIC_LETTERS = r"[\u0621-\u063A\u0641-\u064A]+"
GLUED_LATIN = re.compile(r"[\u0621-\u063A\u0641-\u064A][A-Za-z]|[A-Za-z][\u0621-\u063A\u0641-\u064A]")
TASHKEEL = re.compile(r"[\u064B-\u0652]")

VOICE_AR = {
    "Mona": "أستاذة منى، باحثة سوق هادية وواثقة ومنظّمة.",
    "Mahmoud": "الحاج عمران، تاجر قديم بيقول للمتكلم معاه يا بشمهندس أو يا ابني.",
    "Sherif": "مستر شريف، مهندس عمليات عملي وسريع، وبيدخّل كلمات إنجليزي زي MVP وworkflow.",
    "Manal": "مدام منال، مستشارة مخاطر حادة ومباشرة وبتسخر بهدوء."
}

# Three different examples (different topic), so the model does not copy one of them
NILE_EXAMPLES = [
    ("ما الذي يضمن أن المشترك سيجدد اشتراكه الشهر القادم؟",
     "إيه اللي يضمن إن المشترك هيجدد اشتراكه الشهر الجاي؟"),
    ("الخطة جيدة، لكن لا يوجد دليل كاف على نجاحها.",
     "الخطة حلوة، بس مفيش دليل كفاية على نجاحها."),
    ("كم ستدفع للمدربين، ومن سيتحمل تكلفة الإيجار؟",
     "هتدفع كام للمدربين، ومين اللي هيتحمل تكلفة الإيجار؟")
]


def nile_system(key):
    return f"""إنت بتحوّل جملة من الفصحى لمصري عامي طبيعي، على لسان {VOICE_AR[key]}
القواعد:
- حافظ على نفس المعنى بالظبط، من غير ما تضيف أو تحذف معلومة.
- حافظ على وجهة نظر المتكلم كما هي: لا تحوّل "أنت/ـك/خدمتك/مشروعك" إلى "إحنا/ـنا/خدمتنا/مشروعنا" أو العكس.
- لو الجملة موجهة لصاحب المشروع، خلي المخاطَب هو صاحب المشروع؛ ما تخليش الشخصية تتكلم كأنها شريكة في المشروع.
- حافظ على كل الأرقام.
- لو الجملة سؤال تفضل سؤال، ولو مش سؤال متحوّلهاش لسؤال ومتضيفش سؤال.
- متكتبش كلمات إنجليزي غير الموجودة في الجملة الأصلية، ما عدا MVP وpilot وworkflow وbottleneck.
- اكتب الجملة المحوّلة بس، من غير شرح."""

def nile_stop_ids():
    ids = {tokenizer_nile.eos_token_id}

    # The end-of-turn token of the chat template
    for token in ["<|eot_id|>", "<|end_of_text|>", "<|im_end|>", "<end_of_turn>"]:
        token_id = tokenizer_nile.convert_tokens_to_ids(token)
        if token_id is not None and token_id != tokenizer_nile.unk_token_id:
            ids.add(token_id)

    # Whatever the model's own config says
    config_eos = model_nile.generation_config.eos_token_id
    if config_eos is not None:
        ids.update(config_eos if isinstance(config_eos, list) else [config_eos])

    return [i for i in ids if i is not None]


def nile_generate(messages, temperature=0.0, max_new_tokens=100):
    prompt = tokenizer_nile.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer_nile(prompt, return_tensors="pt").to(model_nile.device)

    if temperature > 0:
        sampling = dict(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        sampling = dict(do_sample=False, temperature=None, top_p=None, top_k=None)

    with generation_lock:
        with torch.inference_mode():
            outputs = model_nile.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer_nile.eos_token_id,
                eos_token_id=nile_stop_ids(),
                **sampling
            )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer_nile.decode(new_tokens, skip_special_tokens=False)

    # Cut at the first special token (the end of the answer)
    text = re.split(r"<\|[^>]*\|>|<end_of_turn>|</s>", text)[0]
    return text.strip()


def content_words(text):
    text = TASHKEEL.sub("", text)
    words = re.findall(ARABIC_LETTERS, text)
    # Remove the leading "ال" so "التطبيق" and "تطبيق" count as the same word
    words = [w[2:] if w.startswith("ال") and len(w) > 4 else w for w in words]
    return {w for w in words if len(w) >= 4}


def keeps_meaning(source, out):
    source_words = content_words(source)

    # Too short to compare safely
    if len(source_words) <= 2:
        return True

    out_words = content_words(out)

    # A word is kept if a similar word exists in the output (يدويا ~ يدوي)
    kept = [
        w for w in source_words
        if any(difflib.SequenceMatcher(None, w, o).ratio() >= 0.75 for o in out_words)
    ]
    return len(kept) / len(source_words) >= 0.2


def nile_problem(source, out):
    if not out:
        return "empty reply"

    # Hard guard against changing the speaker/possessor perspective.
    # نا (our) must not replace ك (your), or vice versa, in common project terms.
    perspective_pairs = [
        ("خدمتك", "خدمتنا"), ("مشروعك", "مشروعنا"), ("فكرتك", "فكرتنا"),
        ("عملاءك", "عملائنا"), ("عملائك", "عملائنا"), ("عميلك", "عميلنا"),
        ("عملك", "عملنا"), ("عندك", "عندنا"), ("ليك", "لينا"),
    ]
    src_norm = _normalize_arabic_text(source) if "_normalize_arabic_text" in globals() else source
    out_norm = _normalize_arabic_text(out) if "_normalize_arabic_text" in globals() else out
    for second_person, first_plural in perspective_pairs:
        if second_person in src_norm and first_plural in out_norm:
            return f"changed founder-directed perspective: {second_person} -> {first_plural}"
        if first_plural in src_norm and second_person in out_norm:
            return f"changed speaker perspective: {first_plural} -> {second_person}"
    if len(out) > 2 * len(source) + 20:
        return "the reply is much longer than the source"
    if "؟" in source and not out.endswith("؟"):
        return "a question must stay a question"
    if numbers_in(out) != numbers_in(source):
        return "numbers changed"
    if not keeps_meaning(source, out):
        return "the meaning changed (too few shared words)"

    for example_source, example_out in NILE_EXAMPLES:
        if source != example_source and difflib.SequenceMatcher(None, out, example_out).ratio() > 0.8:
            return "copied an example"

    # Some words flagged by older versions are normal Egyptian Arabic.
    # Keep the heuristic for strongly formal forms, while exempting common words.
    words = set(re.findall(ARABIC_LETTERS, out))
    formal_only = MSA_WORDS - {"ما", "كيف", "لكن", "هل"}
    if words & formal_only:
        return f"formal Arabic words: {sorted(words & formal_only)}"
    if GLUED_LATIN.search(out):
        return "English letters glued to Arabic"

    allowed = {w.lower() for w in re.findall(r"[A-Za-z]+", source)} | ALLOWED_ENGLISH
    extra = {w for w in re.findall(r"[A-Za-z]+", out) if w.lower() not in allowed}
    if extra:
        return f"new English words: {sorted(extra)}"

    return None


def to_egyptian(key, text, tries=3):
    source = text.strip().replace("?", "؟")

    messages = [{"role": "system", "content": nile_system(key)}]
    for example_source, example_out in NILE_EXAMPLES:
        messages += [
            {"role": "user", "content": example_source},
            {"role": "assistant", "content": example_out}
        ]
    messages.append({"role": "user", "content": source})

    for attempt in range(tries):
        out = nile_generate(messages, temperature=0.0 if attempt == 0 else 0.4)
        out = " ".join(out.strip('" \n').replace("?", "؟").split())

        # A statement must stay a statement
        if "؟" not in source:
            out = drop_questions(out)

        problem = nile_problem(source, out)

        if problem is None:
            return out

    print(f"⚠️ NileChat failed ({problem}). Last output: {out}")
    return source

In [ ]:
app = FastAPI(title="Jow El Tank API")


class IdeaRequest(BaseModel):
    idea: str


class AnalyzeRequest(BaseModel):
    idea: str
    understanding: dict | None = None


def check_api_key(x_api_key: str = Header(None)):
    if x_api_key != JOW_API_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing API key")


def clean_idea(text):
    idea = text.strip()

    if len(idea) < 10:
        raise HTTPException(status_code=422, detail="Idea is too short")
    if len(idea) > 2000:
        raise HTTPException(status_code=422, detail="Idea is too long (max 2000 characters)")

    return idea


def run_agent(name, timings, func, *args):
    start = time.time()
    try:
        result = func(*args)
    except Exception as e:
        raise HTTPException(status_code=502, detail={"failed_agent": name, "error": str(e)})

    timings[name] = round(time.time() - start, 1)
    return result


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/understand", dependencies=[Depends(check_api_key)])
def understand_api(request: IdeaRequest):
    idea = clean_idea(request.idea)
    timings = {}

    understanding = run_agent("Idea Understanding", timings, understand_idea, idea)

    return {"idea": idea, "understanding": understanding}


@app.post("/analyze_sharks", dependencies=[Depends(check_api_key)])
def analyze_sharks_api(request: AnalyzeRequest):
    start = time.time()
    idea = clean_idea(request.idea)
    timings = {}

    # 1. Understanding (use the one from /understand if it was sent)
    understanding = request.understanding
    if understanding is None:
        understanding = run_agent("Idea Understanding", timings, understand_idea, idea)
    else:
        missing = [k for k in UNDERSTAND_KEYS if k not in understanding]
        if missing:
            raise HTTPException(status_code=422, detail=f"understanding is missing keys: {missing}")

         # 2. The four analysts
    mona = run_agent("Mona", timings, mona_market_analysis, idea, understanding)
    mahmoud = run_agent("Mahmoud", timings, mahmoud_finance_analysis, idea, understanding)
    sherif = run_agent("Sherif", timings, sherif_execution_analysis, idea, understanding)
    manal = run_agent("Manal", timings, manal_risk_analysis, idea, understanding)

    # 3. Python: finance, score, decision
    financials = run_agent("Financial Calculator", timings, calculate_financials, mahmoud["assumptions"])
    score = run_agent("Scoring", timings, calculate_final_score, mona, financials, sherif, manal)
    judge = run_agent("Final Judge", timings, final_judge, score, mona, financials, sherif, manal)

    # 4. The board only presents the analysis (the decision does not depend on it)
    board, board_error = None, None
    try:
        board_data = run_agent("Board context", timings, build_board_context, idea, understanding, mona, mahmoud, financials, sherif, manal)
        board = run_agent("Board", timings, board_discussion, board_data)
    except HTTPException as e:
        board_error = e.detail

    timings["total"] = round(time.time() - start, 1)

    return {
        "idea": idea,
        "understanding": understanding,
        "agents": {"Mona": mona, "Mahmoud": mahmoud, "Sherif": sherif, "Manal": manal},
        "financials": financials,
        "board": board,
        "board_error": board_error,
        "score": score,
        "judge": judge,
        "timings": timings
    }

# ------------------------------------------------------------------
# Persistent multi-turn Shark interviews (additive; existing API/jobs stay)
# ------------------------------------------------------------------
interview_sessions = globals().get("interview_sessions", {})
interview_lock = threading.RLock()
SHARK_ORDER = ["Mona", "Mahmoud", "Sherif", "Manal"]
SHARK_TASKS = {
    "Mona": "market and customer discovery, demand evidence, target segment, differentiation, and competitors",
    "Mahmoud": "business model, pricing, revenue, commission, variable costs, fixed costs, and financial assumptions",
    "Sherif": "first-month execution, MVP, workflow, people, operations, and bottlenecks",
    "Manal": "critical assumptions, failure scenarios, risks, mitigations, and unknowns",
}

import re
import unicodedata

import re
import unicodedata


def _is_unknown_answer(message: str) -> bool:
    if not isinstance(message, str) or not message.strip():
        return True

    text = unicodedata.normalize("NFKC", message).strip().lower()
    text = re.sub(r"[\u064B-\u065F\u0670ـ]", "", text)
    text = re.sub(r"[أإآٱ]", "ا", text)
    text = re.sub(r"\s+", " ", text)

    patterns = [
        r"ماعرفش",
        r"معرفش",
        r"مابعرفش",
        r"مش عارف",
        r"مش عارفة",
        r"لسه ما (?:حددت|حددنا|قررت|اخترت|عنديش)",
        r"لسه مش (?:عارف|عارفة|متأكد|محدد|مقرر)",
        r"لسه (?:مقررتش|محددتش|اخترتش|معرفش|مش عارف|مش عارفة)",
        r"ما (?:حددتش|قررتش|اخترتش|عنديش بيانات)",
        r"مش (?:عارف|عارفة|متأكد|محدد|مقرر)",
        r"مش عارف (?:لسه|لحد دلوقتي)",
        r"لا اعرف",
        r"لا توجد بيانات",
        r"غير معروف",
        r"غير محدد",
        r"لم يتم تحديد",
        r"محتاج (?:اجمع|ابحث|اعرف|اقارن|اختبر|اجرب)",
        r"عايز اقارن",
        r"لسه بفكر",
        r"لسه مقررتش",
        r"لسه محددتش",
        r"محتاج اختبر الفكرة",
        r"محتاج اجرب الفكرة",
        r"محتار بين",
        r"قيد الدراسة",
    ]

    return any(re.search(pattern, text) for pattern in patterns)


def _session_context(session):
    return {"confirmed_facts":session["confirmed_facts"], "assumptions":session["assumptions"],
            "unknowns":session["unknowns"], "unanswered_important_questions":session["unanswered_important_questions"],
            "conversation_history":session["history"], "progress":session["progress"]}


def _start_discussion(session):
    if session["discussion_rounds"] >= 2:
        session["phase"] = "final_ready"
        return {
            "speaker": "Jow El Tank",
            "message": "انتهى النقاش. تقدر تطلب التقييم النهائي."
        }

    round_number = session["discussion_rounds"] + 1

    system_prompt = (
        "You run a short discussion between four startup sharks. "
        "Mona = market and competition, Mahmoud = finance, "
        "Sherif = operations, Manal = risk and strategy. "
        "Use only shared data. Never invent facts or numbers. "
        "Each shark must produce exactly one short declarative statement, "
        "not a question. Return exactly four turns in this order: "
        "Mona, Mahmoud, Sherif, Manal. "
        "Then choose one asker and ask one short, new question. "
        "Avoid repeating previous questions. "
        "Return ONLY valid JSON with keys turns, asker, question."
    )

    shared = {
        "round": round_number,
        "idea": session.get("idea", ""),
        "confirmed_facts": [
            str(f)[:200]
            for f in session.get("confirmed_facts", [])[-8:]
        ],
        "unknowns": session.get("unknowns", [])[-6:],
        "unanswered_important_questions":
            session.get("unanswered_important_questions", [])[-6:],
        "conversation": _compact_history(session),
        "previous_questions": _interviewer_questions(session),
    }

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": json.dumps(shared, ensure_ascii=False)
        },
    ]

    data = None

    # Retry malformed or incomplete AI responses.
    for attempt in range(2):
        raw = qwen_generate(
            messages, max_new_tokens=600, temperature=0.0
        )
        print("[DEBUG] Discussion raw response:")
        print(str(raw)[:3000])
        
        candidate = parse_agent_result(raw)
        print("[DEBUG] Parsed type:", type(candidate).__name__)
        print("[DEBUG] Parsed keys:",
              list(candidate.keys()) if isinstance(candidate, dict) else None)

        if isinstance(candidate, dict):
            turns = candidate.get("turns")
            asker = str(candidate.get("asker", "")).strip()
            question = _single_question(candidate.get("question", ""))

            
            valid_turns = (
                isinstance(turns, list)
                and len(turns) == 4
                and all(
                    (
                        isinstance(t, dict)
                        and str(t.get("speaker", "")).strip() in SHARK_ORDER
                        and str(t.get("message", "")).strip()
                    )
                    or (
                        isinstance(t, str)
                        and ":" in t
                        and t.split(":", 1)[0].strip() in SHARK_ORDER
                        and t.split(":", 1)[1].strip()
                    )
                    for t in turns
                )
            )


            if (
                valid_turns
                and asker in SHARK_ORDER
                and question
            ):
                data = candidate
                break

        print(
            f"[DEBUG] Discussion invalid response, "
            f"attempt {attempt + 1}: {str(raw)[-300:]!r}"
        )
        messages += [
            {"role": "assistant", "content": str(raw or "")},
            {
                "role": "user",
                "content": (
                    "Return complete valid JSON with exactly four turns, "
                    "one for each speaker in the required order, plus "
                    "asker and question. Each message must be a short "
                    "statement, not a question. No Markdown."
                )
            }
        ]

    if data is None:
        # Keep the app alive even if generation fails.
        session["discussion_rounds"] += 1
        session["phase"] = (
            "final_ready"
            if session["discussion_rounds"] >= 2
            else "discussion"
        )
        return {
            "speaker": "Jow El Tank",
            "message": (
                "واجهنا مشكلة في توليد النقاش. "
                "تقدر تكمل أو تطلب التقييم النهائي."
            ),
            "discussion": []
        }

    turns = data["turns"]
    asker = str(data["asker"]).strip()
    question = _single_question(data["question"])

    spoken = {}

    for turn in turns:
        if isinstance(turn, dict):
            key = str(turn.get("speaker", "")).strip()
            message = " ".join(
                str(turn.get("message", "")).split()
            )
        elif isinstance(turn, str) and ":" in turn:
            key, message = turn.split(":", 1)
            key = key.strip()
            message = " ".join(message.split())
        else:
            continue
    
        # Ignore questions instead of accidentally turning them into statements.
        if "؟" in message or "?" in message:
            continue
    
        if key in SHARK_ORDER and key not in spoken and message:
            spoken[key] = to_egyptian(key, message)

    # Safe fallback for missing or unusable shark statements.
    fallback_statements = {
        "Mona": "لازم نتأكد إن العملاء محتاجين الخدمة ومستعدين يجربوها.",
        "Mahmoud": "لازم نعرف تكلفة كل حجز وهل العمولة هتغطيها.",
        "Sherif": "لازم نتأكد إن المواعيد والحجوزات هتتحدث بدقة.",
        "Manal": "لازم نختبر المخاطر باختبار صغير قبل التوسع."
    }

    discussion = []
    for key in SHARK_ORDER:
        message = spoken.get(key) or fallback_statements.get(key)
        if message:
            discussion.append({
                "speaker": key,
                "message": message
            })

    question = _single_question(to_egyptian(asker, question))

    if _question_is_bad(session, question, shark=asker):
        question = _get_new_question(
            session, asker, question, max_attempts=1
        ) or _fallback_question(session, asker)

    # Never return an empty or known-invalid question.
    if not question or _question_is_bad(session, question, shark=asker):
        question = (
            "إيه أهم حاجة محتاج تتأكد منها قبل ما توسع المشروع؟"
        )

    session["discussion_rounds"] += 1
    session["phase"] = "discussion"

    for turn in discussion:
        session["history"].append({
            "speaker": turn["speaker"] + " (discussion)",
            "text": turn["message"]
        })

    return {
        "speaker": asker,
        "message": question,
        "discussion": discussion
    }
def _prompt_context(session, compact=False):
    # Short version of the interview for the analysts' prompts (the API output stays full).
    # The founder's answers are already inside confirmed_facts, so the history is not repeated.
    if compact:
        return {
            "confirmed_facts": [str(f)[:200] for f in session["confirmed_facts"]][-6:],
            "unknowns": session["unknowns"][-4:],
        }
    return {
        "confirmed_facts": [str(f)[:200] for f in session["confirmed_facts"]][-10:],
        "assumptions": session["assumptions"][-6:],
        "unknowns": session["unknowns"][-6:],
        "unanswered_important_questions": session["unanswered_important_questions"][-6:],
        "progress": {
            k: {**v, "summary": str(v.get("summary", ""))[:250]}
            for k, v in session["progress"].items()
        },
    }


def _build_current_analysis(session):
    idea = session["idea"]
    understanding = dict(session["understanding"])
    understanding["shared_interview_context"] = _prompt_context(session)

    # Mona already carries the web sources, so she gets the shortest context
    mona_understanding = dict(session["understanding"])
    mona_understanding["shared_interview_context"] = _prompt_context(session, compact=True)
    mona = mona_market_analysis(idea, mona_understanding)
    mahmoud = mahmoud_finance_analysis(idea, understanding)
    financials = calculate_financials(mahmoud["assumptions"])
    sherif = sherif_execution_analysis(idea, understanding)
    manal = manal_risk_analysis(idea, understanding)
    score = calculate_final_score(mona, financials, sherif, manal)
    judge = final_judge(score, mona, financials, sherif, manal)
    try:
        board = board_discussion(build_board_context(idea, understanding, mona, mahmoud, financials, sherif, manal))
    except Exception as exc:
        board = {"error":str(exc)}
    return {"idea":idea,"understanding":understanding,
            "agents":{"Mona":mona,"Mahmoud":mahmoud,"Sherif":sherif,"Manal":manal},
            "financials":financials,"score":score,"judge":judge,"board":board,
            "interview_context":_session_context(session)}

class SessionStartRequest(BaseModel):
    idea: str
class SessionTurnRequest(BaseModel):
    message: str
    early_evaluation: bool = False

@app.post("/sessions", dependencies=[Depends(check_api_key)])
def create_interview_session(request: SessionStartRequest):
    idea = clean_idea(request.idea)
    understanding = run_agent("Idea Understanding", {}, understand_idea, idea)
    unclear = any(str(understanding.get(k,"")).strip() in ("","مش محدد في الفكرة")
                  for k in ("problem","solution","target_customer"))
    session_id = str(uuid.uuid4())
    session = {"session_id":session_id,"idea":idea,"understanding":understanding,
        "phase":"clarification" if unclear else "interview","current_shark":"Mona",
        "history":[],"confirmed_facts":[],"assumptions":[],
        "unknowns":list(understanding.get("missing_information",[])),
        "unanswered_important_questions":list(understanding.get("missing_information",[])),
        "progress":{k:{"status":"pending","summary":"","turns":0} for k in SHARK_ORDER},
        "discussion_rounds":0,"last_question":None,"created_at":time.time(),"final_evaluation":None}
    with interview_lock: interview_sessions[session_id] = session
    question = _ask_next(session)
    session["last_question"] = question
    return {"session_id":session_id,"phase":session["phase"],"question":question,"progress":session["progress"]}

@app.get("/sessions/{session_id}", dependencies=[Depends(check_api_key)])
def get_interview_session(session_id: str):
    with interview_lock:
        if session_id not in interview_sessions: raise HTTPException(status_code=404,detail="Interview session not found")
        s = interview_sessions[session_id]
        return {"session_id":session_id,"phase":s["phase"],"idea":s["idea"],"progress":s["progress"],
                "context":_session_context(s),"last_question":s["last_question"],
                "discussion_rounds":s["discussion_rounds"],"final_evaluation":s["final_evaluation"]}

@app.post("/sessions/{session_id}/turn", dependencies=[Depends(check_api_key)])
def interview_turn(session_id: str, request: SessionTurnRequest):
    with interview_lock:
        if session_id not in interview_sessions: raise HTTPException(status_code=404,detail="Interview session not found")
        s = interview_sessions[session_id]
        msg = request.message.strip()
        if not msg: raise HTTPException(status_code=422,detail="message cannot be empty")
        if s["phase"] in ("final_ready","complete") and not request.early_evaluation:
            raise HTTPException(status_code=409,detail="Interviews finished; evaluate or request early evaluation")
        previous = s["last_question"] or {"speaker":"Jow El Tank","message":""}
        s["history"].append({"speaker":previous["speaker"],"text":previous["message"]})
        s["history"].append({"speaker":"Founder","text":msg})
        if _is_unknown_answer(msg):
            s["unknowns"].append(f"Founder does not know yet: {previous['message']}")
            s["unanswered_important_questions"].append(previous["message"])
        else:
            s["confirmed_facts"].append(f"Q ({previous['speaker']}): {previous['message']} | Founder's answer: {msg}")
        if request.early_evaluation:
            s["phase"] = "final_ready"
            return {"session_id":session_id,"phase":s["phase"],"message":"Early evaluation requested; unresolved items will remain labelled."}
        if s["phase"] == "clarification":
            s["idea"] += "\nFounder clarification: " + msg
            s["understanding"] = run_agent("Idea Understanding", {}, understand_idea, s["idea"])
            unclear = any(str(s["understanding"].get(k,"")).strip() in ("","مش محدد في الفكرة")
                          for k in ("problem","solution","target_customer"))
            if not unclear: s["phase"] = "interview"
        elif s["phase"] == "interview":
            s["progress"][s["current_shark"]]["turns"] += 1
        elif s["phase"] == "discussion":
            if s["discussion_rounds"] >= 2: s["phase"] = "final_ready"
        q = _ask_next(s)
        s["last_question"] = q
        return {"session_id":session_id,"phase":s["phase"],"question":q,"progress":s["progress"]}

@app.post("/sessions/{session_id}/evaluate", dependencies=[Depends(check_api_key)])
def evaluate_interview_session(session_id: str, early: bool = False):
    with interview_lock:
        if session_id not in interview_sessions: raise HTTPException(status_code=404,detail="Interview session not found")
        s = interview_sessions[session_id]
        if s["phase"] not in ("final_ready","complete") and not early:
            raise HTTPException(status_code=409,detail="Interviews are not complete; pass ?early=true to evaluate with unresolved items labelled")
        if early and s["phase"] not in ("complete",):
            s["phase"] = "final_ready"
        result = _build_current_analysis(s)
        s["final_evaluation"] = result
        s["phase"] = "complete"
        return result


In [ ]:
SHARK_STYLE = {
    "Mona": (
        "Focus on customers, demand, competitors, and what makes "
        "the idea different. Use everyday words."
    ),
    "Mahmoud": (
        "Focus on how the project earns and spends money. "
        "Use simple words and practical examples."
    ),
    "Sherif": (
        "Focus on the first steps, how the service works, "
        "and who does each task. Prefer small manual steps."
    ),
    "Manal": (
        "Your role is to identify risks, possible failures, and backup plans. "
        "Ask what might go wrong in THIS specific idea and how the founder "
        "would respond: losing money, quality or trust problems, delays, "
        "a key partner leaving, complaints, or running out of money, "
        "whichever applies to the idea. "
        "Do not ask general questions about customer demand, customer numbers, "
        "competitors, or marketing; these belong to Mona. "
        "Ask only one short question."
    ),
}


In [ ]:
# Jow El Tank — consolidated interview-flow fix
# This cell intentionally runs before the server-start cell. It does not start/restart FastAPI or ngrok.
import re
import unicodedata


def _normalize_arabic_text(text):
    text = unicodedata.normalize("NFKC", str(text)).strip().lower()
    text = re.sub(r"[\u064B-\u065F\u0670ـ]", "", text)
    text = re.sub(r"[أإآٱ]", "ا", text)
    return re.sub(r"\s+", " ", text)





# Generic fallback questions (Egyptian Arabic). They do not depend on the type of business.
SHARK_FALLBACKS = {
    "Mona": [
        "إيه اللي هيخلي العميل يسيب اللي بيستخدمه دلوقتي ويجرب فكرتك؟",
        "مين أول مجموعة عملاء هتبدأ بيهم بالظبط؟",
        "هتوصل لأول عملاء إزاي؟",
        "إزاي هتتأكد إن الناس محتاجة الفكرة دي قبل ما تصرف عليها؟",
    ],
    "Mahmoud": [
        "هتكسب فلوس من فين بالظبط؟",
        "إيه أكبر تكلفة هتصرفها كل شهر في البداية؟",
        "هتحدد السعر اللي هتاخده من العميل إزاي؟",
        "كل عملية هتكلفك كام تقريبًا؟",
    ],
    "Sherif": [
        "هتعمل إيه في أول يوم تشغيل بالظبط؟",
        "إيه اللي ممكن يتعمل يدوي في أول شهر؟",
        "مين هيعمل كل مهمة في أول نسخة؟",
        "إيه أول حاجة ممكن تعطل التشغيل في أول شهر؟",
    ],
    "Manal": [
        "لو أهم افتراض في فكرتك طلع غلط، هتعمل إيه؟",
        "لو خسرت فلوس في أول شهرين، هتتصرف إزاي؟",
        "إيه أخطر حاجة ممكن تخلي العميل يسيب الخدمة؟",
        "لو العملاء مدفعوش، إيه خطتك البديلة؟",
    ],
}


def _interviewer_questions(session):
    # Only real questions asked by the sharks (statements and founder answers are skipped)
    return [
        item.get("text", "").strip()
        for item in session.get("history", [])
        if item.get("speaker") not in ("Founder", "Clarifier")
        and item.get("text", "").strip().endswith("؟")
    ]


def _compact_history(session, limit=10, max_chars=200):
    # Long answers are cut, so the prompt never grows past the model's input limit
    return [
        f"{item.get('speaker', '')}: {str(item.get('text', ''))[:max_chars]}"
        for item in session.get("history", [])[-limit:]
    ]


def _single_question(text):
    # Keep only the first question; return "" if there is no question at all
    text = " ".join(str(text or "").replace("?", "؟").split())
    if "؟" not in text:
        return ""
    return text[:text.index("؟") + 1]


def _known_text(session):
    founder = " ".join(
        item.get("text", "")
        for item in session.get("history", [])
        if item.get("speaker") == "Founder"
    )
    understanding = json.dumps(session.get("understanding", {}), ensure_ascii=False)
    return session.get("idea", "") + " " + understanding + " " + founder


# Words that show a question belongs to another shark (matched on normalized text)
MONEY_WORDS = ["ربح", "ارباح", "عموله", "ايراد", "تكلف", "سعر", "اسعار", "يكسب", "هتكسب", "فلوس"]
MARKET_WORDS = ["منافس", "تسويق", "حجم السوق"]
OUT_OF_DOMAIN = {
    "Mona": MONEY_WORDS,
    "Sherif": MONEY_WORDS,
    "Mahmoud": MARKET_WORDS,
    "Manal": MARKET_WORDS,
}


def _out_of_domain(shark, question):
    text = _normalize_question_for_dedup(question or "").replace("ة", "ه")
    return any(word in text for word in OUT_OF_DOMAIN.get(shark, []))


def _question_is_bad(session, question, check_semantic=True, shark=None):
    if not question or not question.endswith("؟") or len(question) > 200:
        return True
    if shark and _out_of_domain(shark, question):
        print(f"[DEBUG] {shark} question rejected: belongs to another shark")
        return True
    if _is_generic_interview_question(question) or _is_meta_interview_question(question):
        return True
    # A number nobody said
    if numbers_in(question) - numbers_in(_known_text(session)):
        return True
    if _question_was_asked(session, question):
        return True
    if check_semantic:
        return _is_semantic_duplicate(question, _interviewer_questions(session))
    return False


def _fallback_question(session, shark):
    for question in SHARK_FALLBACKS[shark]:
        if not _question_is_bad(
            session, question,
            check_semantic=True, shark=shark
        ):
            return question

    # Never return a known duplicate.
    return ""


def _question_was_asked(session, question):
    """Detect repeated questions and print why they matched."""

    target = _normalize_arabic_text(question)
    if not target:
        return False

    previous_questions = _interviewer_questions(session)

    print("[DEBUG] Checking question:", repr(question))
    print("[DEBUG] Previous interviewer questions:", previous_questions[-10:])

    for previous in previous_questions:
        if _normalize_arabic_text(previous) == target:
            print("[DEBUG] Exact duplicate:", repr(previous))
            return True

    for previous in previous_questions:
        if _same_business_topic(previous, question):
            print("[DEBUG] Same-topic match:", repr(previous))
            return True

    unresolved = session.get("unanswered_important_questions", [])
    for previous in unresolved:
        if _same_business_topic(previous, question):
            print("[DEBUG] Matches unresolved topic:", repr(previous))
            return True

    print("[DEBUG] No duplicate found.")
    return False


def _is_generic_interview_question(question):
    """Reject vague questions that could apply to almost any startup."""
    normalized = _normalize_question_for_dedup(question or "")

    # Normalize common Arabic spelling variants for matching.
    normalized = normalized.replace("ة", "ه")

    generic_patterns = [
        "ايه اهم حاجه لسه محتاج تتاكد منها",
        "ايه اهم حاجه محتاج تتاكد منها",
        "ايه اكبر تحدي في مشروعك",
        "ايه اكبر عقبه متوقع تقابلك",
        "ايه اول خطوه هتعملها",
    ]

    for pattern in generic_patterns:
        pattern_normalized = _normalize_question_for_dedup(pattern)
        pattern_normalized = pattern_normalized.replace("ة", "ه")

        if pattern_normalized in normalized:
            return True

    return False



def _get_new_question(session, shark, question, max_attempts=2):
    """Generate a question on a genuinely new topic, in the shark's voice."""
    p = PERSONAS[shark]
    previous_questions = _interviewer_questions(session)

    system_prompt = (
        f"You are {p['name']} in Jow El Tank. Your character: {p['character']} "
        f"Your responsibility: {SHARK_TASKS[shark]}. "
        "Write exactly ONE short question (under 20 words) in clear, simple "
        "Modern Standard Arabic, in your character's voice "
        "(another model turns it into Egyptian dialect). "
        "Ask only inside your responsibility; money, profit and commission belong to Mahmoud. "
        "Read the conversation first. Never repeat or paraphrase a question "
        "that was already asked. Pick ONE specific unresolved decision inside "
        "your responsibility that the founder has not answered yet. "
        "Do not invent facts or numbers. "
        "Return only the question, without explanation or JSON."
    )

    for attempt in range(max_attempts):
        messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "idea": session.get("idea", ""),
                        "understanding": session.get("understanding", {}),
                        "conversation": _compact_history(session),
                        "unknowns": session.get("unknowns", [])[-6:],
                        "previous_questions": previous_questions,
                        "question_to_replace": question,
                    },
                    ensure_ascii=False,
                ),
            },
        ]

        try:
            candidate = qwen_generate(
                messages,
                max_new_tokens=100,
                temperature=0.3,
                enable_thinking=False,
            ).strip().strip('"')
        except Exception as exc:
            print(f"[DEBUG] Question generation failed: {exc}")
            continue

        candidate = _single_question(candidate)
        if not candidate:
            continue

        # One conversion to Egyptian, then check the final text
        candidate = _single_question(to_egyptian(shark, candidate))

        if not _question_is_bad(session, candidate, shark=shark):
            return candidate

    return ""


def _updated_shark_summary(session, shark, model_summary=""):
    """Build a current summary from founder statements and unresolved items."""
    model_summary = str(model_summary or "").strip()
    if shark != "Mona":
        return model_summary or "لم يكتمل الملخص بعد؛ ستُحدّثه المقابلة مع ورود معلومات جديدة."

    founder_answers = []
    for item in session.get("history", []):
        if item.get("speaker") == "Founder":
            answer = str(item.get("text", "")).strip()
            if answer and answer not in founder_answers:
                founder_answers.append(answer)
    founder_answers = founder_answers[-5:]

    unknowns = []
    for item in session.get("unknowns", []):
        value = str(item).strip()
        if value and value not in unknowns:
            unknowns.append(value)
    unknowns = unknowns[-5:]

    parts = []
    if model_summary:
        parts.append("ملخص التحليل الحالي: " + model_summary)
    if founder_answers:
        parts.append("أحدث إجابات المؤسس: " + "؛ ".join(founder_answers))
    if unknowns:
        parts.append("نقاط لم تُحسم بعد: " + "؛ ".join(unknowns))
    return " | ".join(parts) or "لم تتوفر معلومات كافية لملخص موثوق حتى الآن."


def _interview_instruction(session, shark):
    p = PERSONAS[shark]
    examples = "\n".join(f"- {x}" for x in p["examples"])
    asked = "\n".join(f"- {q}" for q in _interviewer_questions(session)[-12:]) or "(none yet)"

    shared = {
        "confirmed_facts": [str(f)[:200] for f in session.get("confirmed_facts", [])[-8:]],
        "assumptions": session.get("assumptions", [])[-6:],
        "unknowns": session.get("unknowns", [])[-6:],
        "unanswered_important_questions": session.get("unanswered_important_questions", [])[-6:],
    }
    shared_text = json.dumps(shared, ensure_ascii=False)
    understanding_text = json.dumps(session.get("understanding", {}), ensure_ascii=False)

    return f"""You are {p['name']} in Jow El Tank, talking directly to the founder in a live conversation.
YOUR ROLE: {p['role']}
YOUR CHARACTER: {p['character']}
YOUR RESPONSIBILITY: {SHARK_TASKS[shark]}.

How you talk (voice only, never reuse these exact words):
{examples}

LANGUAGE:
- Write the question in clear, simple Modern Standard Arabic (another model turns it into Egyptian dialect).
- Keep your character: the way you address the founder and the style you naturally use.
- English is allowed ONLY for these words if your character uses them: MVP, pilot, workflow, bottleneck.

YOUR MAIN JOB:
Help the founder discover whether this specific business can work in reality.
Choose the single most important unresolved decision inside your responsibility,
based on the idea, the conversation so far, and the founder's latest answer.

WHO ASKS WHAT (very important):
- Mona: customers, demand, competitors, differentiation, how customers find the service.
- Mahmoud: how the business earns money, pricing, commission, costs.
- Sherif: how the first version runs in the first month, people, manual steps.
- Manal: what could make the idea fail and the backup plan.
- The UNDERSTANDING and SHARED CONTEXT may list missing items that belong to ANOTHER shark.
  Ignore them. Ask only about items inside YOUR responsibility.
  Never ask about money, profit, or commission unless you are Mahmoud.

QUESTIONS ALREADY ASKED (never repeat them or ask them in other words):
{asked}

QUESTION PRIORITY:
- Review the shared context and the conversation before choosing a question.
- Never ask about a decision that the founder has already answered clearly.
- If a previous answer leaves one specific part unresolved, ask about that
  missing part instead of restarting the broader topic.
- Prefer the highest-impact unanswered question that is supported by the
  founder's answers and the business idea.
- Do not ask the founder to explain something again if they already explained it.

1. Understand the actual customer, problem, and proposed solution.
2. Test the riskiest assumptions that could make the idea fail.
3. Ask for practical evidence or a small real-world experiment.
4. Explore pricing, costs, revenue, operations, or risks when relevant to your responsibility.
5. Ask about technical details only when they are necessary to solve a clearly identified business problem.

QUESTION QUALITY RULES:
- For financial questions, ask for one concrete value or calculation.
- Never ask vague questions like "what amount determines success?".
- If break-even orders or monthly expenses were already discussed,
  ask about a different missing financial detail.
- Every question must refer directly to the founder's actual business idea.
- Prefer specific, consequential questions over broad or hypothetical ones.
- Never ask generic questions such as "What else do you need to know?"
  or "What is the most important thing before starting?"
- Do not introduce features, technologies or numbers that neither the
  founder nor the idea mentioned.
- Do not repeat a previous question using synonyms or a different sentence structure.
  A different wording is not a different question if it seeks the same answer.
- Do not ask another shark's main question unless a real dependency requires it.
- Ask only ONE question, under 20 words, ending with a question mark.
- Never assume customers will buy, that the idea will work, or that evidence exists.

USE THE FOUNDER'S ANSWERS:
- Treat explicit founder statements as authoritative; founder corrections override older information.
- Use the latest answer to decide the next question.
- If the founder does not know, do not repeat the same question. Move to a
  different missing decision, or help him find a practical way to find out
  (for example a small test with a few potential customers).
- A proposed experiment is not proof that the idea works.
- Never turn your own suggestions into confirmed facts.
- Do not invent facts, numbers, customer preferences, or market evidence.

READINESS:
- Decide ready=true only when the important questions inside your responsibility
  have been answered well enough for a useful assessment.
- Do not mark ready=true merely because a fixed number of questions was asked.
- If ready=true, question must be an empty string.
- Otherwise, question must contain exactly one useful, non-repeated question.

OUTPUT RULES:
- Return ONLY valid JSON with exactly these keys:
  ready (boolean), question (string),
  assumptions (array), unknowns (array),
  unanswered_important_questions (array), summary (string).
- Never output or reproduce the conversation history.
- Use at most 3 short items in each array; each item is one short sentence.
- Keep summary under 50 words and make it reflect the latest answers.
- If a list has no supported items, return an empty array.
- Ensure all JSON strings are valid and properly escaped.

SHARED CONTEXT:
{shared_text}

IDEA:
{session.get('idea', '')}

UNDERSTANDING:
{understanding_text}

SHARK-SPECIFIC GUIDANCE:
{SHARK_STYLE[shark]}"""


def _ask_next(session):
    if session["phase"] == "clarification":
        messages = [
            {"role": "system", "content": "Clarify the business idea without inventing facts. Ask exactly one short question in Egyptian Arabic. Return only the question."},
            {"role": "user", "content": json.dumps({
                "idea": session.get("idea", ""),
                "understanding": session.get("understanding", {}),
                "history": session.get("history", [])[-10:],
            }, ensure_ascii=False)},
        ]
        question = qwen_generate(messages, max_new_tokens=180, temperature=0.0).strip()
        if not question:
            raise ValueError("Clarifier returned an empty question")
        return {"speaker": "Clarifier", "message": question}

    if session["phase"] == "interview":
        shark = session["current_shark"]

        messages = [
            {"role": "system", "content": _interview_instruction(session, shark)},
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "idea": session.get("idea", ""),
                        "conversation": _compact_history(session),
                        "task": (
                            "Return the required JSON object only. Ask one new, "
                            "unanswered question inside your responsibility."
                        ),
                    },
                    ensure_ascii=False,
                ),
            },
        ]

        print(f"[DEBUG] Shark: {shark} | history entries: {len(session.get('history', []))}")

       
        required = [
            "ready", "question", "assumptions",
            "unknowns", "unanswered_important_questions"
        ]
        data, last_raw = None, ""

        for attempt in range(2):
            last_raw = qwen_generate(
                messages, max_new_tokens=500, temperature=0.0
            )
            candidate = parse_agent_result(last_raw)

            valid = (
                isinstance(candidate, dict)
                and isinstance(candidate.get("ready"), bool)
                and all(key in candidate for key in required)
                and isinstance(candidate.get("question"), str)
            )

            if valid:
                # summary is optional in the model response.
                candidate.setdefault("summary", "")
                data = candidate
                break

            print(
                f"[DEBUG] {shark} invalid JSON on attempt {attempt + 1}. "
                f"Tail: {str(last_raw)[-300:]!r}"
            )
            messages = messages + [
                {"role": "assistant", "content": str(last_raw) or "(empty)"},
                {
                    "role": "user",
                    "content": (
                        "Return one complete valid JSON object. Include "
                        "ready (boolean), question (string), assumptions "
                        "(array), unknowns (array), "
                        "unanswered_important_questions (array), and "
                        "summary (string). No Markdown."
                    ),
                },
            ]

        if data is None:
            raise ValueError(
                f"{shark} returned invalid JSON after retry. "
                f"Output tail: {str(last_raw)[-400:]}"
            )


        # A shark must ask at least one question before it can be ready
        if session["progress"][shark].get("turns", 0) == 0:
            data["ready"] = False

        # Only founder answers (recorded by the /turn endpoint) become confirmed facts.
        # Model-proposed claims stay in assumptions/unknowns until the founder confirms them.
        for key in ("assumptions", "unknowns", "unanswered_important_questions"):
            values = data.get(key, [])
            if not isinstance(values, list):
                values = [values]

            clean_values = [str(value).strip() for value in values if str(value).strip()]

            if key in ("unknowns", "unanswered_important_questions"):
                session[key] = _merge_unique_questions(session.get(key, []), clean_values)
            else:
                session[key] = list(dict.fromkeys(session.get(key, []) + clean_values))

        model_summary = str(data.get("summary", "")).strip()
        summary = _updated_shark_summary_v2(session, shark, model_summary)
        old_progress = session["progress"].get(shark, {})
        session["progress"][shark] = {
            "status": "complete" if data["ready"] else "in_progress",
            "summary": summary,
            "turns": old_progress.get("turns", 0),
        }

        if data["ready"]:
            index = SHARK_ORDER.index(shark)
            if index < len(SHARK_ORDER) - 1:
                session["current_shark"] = SHARK_ORDER[index + 1]
                session["progress"][session["current_shark"]]["status"] = "in_progress"
                return _ask_next(session)

            session["phase"] = "final_ready"
            return {
                "speaker": "Jow El Tank",
                "message": "انتهت مقابلات المستثمرين الأربعة. تقدر تطلب التقييم النهائي."
            }

        # 1. Keep one question, convert it to Egyptian once
        raw_question = str(data.get("question", "")).strip()
        question = _single_question(raw_question)
        if question:
            question = _single_question(to_egyptian(shark, question))

       
        # 2. Reject repeated or invalid questions and request replacements
        if _question_is_bad(session, question, shark=shark):
            question = _get_new_question(
                session, shark, question or raw_question, max_attempts=2
            )
            question = _single_question(question or "")
            if question:
                question = _single_question(to_egyptian(shark, question))

            # Validate the replacement too; never trust generated output blindly.
            if _question_is_bad(session, question, shark=shark):
                question = ""

        # 3. Try a fallback, but validate it with the same rules
        if not question:
            question = _fallback_question(session, shark)

        if question and _question_is_bad(session, question, shark=shark):
            question = ""

        # 4. No valid question remains: finish this shark safely
        # rather than returning an empty message or repeating an old question.
        if not question:
            print(
                f"[DEBUG] {shark}: no valid new questions remain; "
                "moving to the next interviewer."
            )
            session["progress"][shark]["status"] = "complete"

            index = SHARK_ORDER.index(shark)
            if index < len(SHARK_ORDER) - 1:
                next_shark = SHARK_ORDER[index + 1]
                session["current_shark"] = next_shark
                session["progress"][next_shark]["status"] = "in_progress"
                return _ask_next(session)

            session["phase"] = "final_ready"
            return {
                "speaker": "Jow El Tank",
                "message": "انتهت مقابلات المستثمرين الأربعة. تقدر تطلب التقييم النهائي."
            }

        return {"speaker": shark, "message": question}


    if session["phase"] == "discussion":
    session["phase"] = "final_ready"
    return {
        "speaker": "Jow El Tank",
        "message": "انتهت مقابلات المستثمرين الأربعة. تقدر تطلب التقييم النهائي."
    }
    if session["phase"] in ("final_ready", "complete"):
        return {"speaker": "Jow El Tank", "message": "التقييم النهائي متاح عند الطلب."}
    return {"speaker": "Jow El Tank", "message": "لا يوجد سؤال متاح في المرحلة الحالية."}


# Consolidated helpers: deduplication and current summary

def _normalize_question_for_dedup(value):
    import re
    import unicodedata

    text = unicodedata.normalize(
        "NFKC", str(value or "")
    ).strip().lower()

    # Remove known prefixes before removing punctuation.
    prefixes = (
        "founder does not know yet:",
        "هل ",
        "هو ",
        "هي ",
    )

    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if text.startswith(prefix):
                text = text[len(prefix):].strip()
                changed = True

    text = re.sub(r"[\u064B-\u065F\u0670ـ]", "", text)
    text = re.sub(r"[أإآٱ]", "ا", text)
    text = re.sub(r"[؟?!.,،:;]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text


def _same_business_topic(question_a, question_b):
    """Detect repeated questions about the same specific business detail."""
    import re

    def normalize(text):
        text = _normalize_arabic_text(str(text or ""))
        text = re.sub(r"[؟?!.,،:;]", " ", text)
        return " ".join(text.split())

    a = normalize(question_a)
    b = normalize(question_b)

    if not a or not b:
        return False

    if a == b:
        return True

    # Compare specific subjects instead of broad business categories.
    topic_groups = {
        "commission_rate": [
            "نسبه العموله", "كام عموله", "نسبه من البيع",
            "عموله من كل طلب", "عموله التطبيق",
        ],
        "delivery_cost": [
            "تكلفه التوصيل", "تكلفه الشحن",
            "مصاريف التوصيل", "تكلفه المندوب",
        ],
        "delivery_time": [
            "مده التوصيل", "وقت التوصيل",
            "التوصيل بياخد", "يوصل الطلب",
        ],
        "net_profit": [
            "صافي الربح", "هامش الربح",
            "يتبقى للتطبيق", "ربح التطبيق من كل",
        ],
        "break_even": [
            "نقطه التعادل", "تغطيه المصاريف",
            "عدد الاوردرات المطلوب", "يغطي مصاريف التطبيق",
        ],
        "monthly_costs": [
            "المصاريف الشهريه", "تكلفه التشغيل الشهريه",
            "مصاريف التطبيق الشهريه",
        ],
        "customer_acquisition": [
            "اول 20 عميل", "اكتساب العملاء",
            "جذب العملاء", "الوصول للعملاء",
        ],
    }

    topics_a = {
        topic for topic, keywords in topic_groups.items()
        if any(keyword in a for keyword in keywords)
    }
    topics_b = {
        topic for topic, keywords in topic_groups.items()
        if any(keyword in b for keyword in keywords)
    }

    # Only reject when both questions clearly target the same detail.
    return bool(topics_a & topics_b)

def _merge_unique_questions(existing, incoming):
    """Merge unresolved questions and remove semantic duplicates."""
    import re

    result = []

    for value in list(existing or []) + list(incoming or []):
        value = str(value or "").strip()
        if not value:
            continue

        # Remove internal English prefix.
        value = re.sub(
            r"^Founder does not know yet:\s*",
            "",
            value,
            flags=re.IGNORECASE,
        ).strip()

        normalized = _normalize_question_for_dedup(value)

        if not normalized:
            continue

        # Skip exact duplicates and questions about the same business topic.
        is_duplicate = any(
            normalized == _normalize_question_for_dedup(previous)
            or _same_business_topic(value, previous)
            for previous in result
        )

        if not is_duplicate:
            result.append(value)

    return result

def _is_semantic_duplicate(candidate, previous_questions):
    # One model call for all recent questions (not one call per question)
    candidate = str(candidate or "").strip()
    if not candidate:
        return True

    previous = [str(q or "").strip() for q in previous_questions if str(q or "").strip()][-8:]
    if not previous:
        return False

    for old in previous:
        if _normalize_arabic_text(candidate) == _normalize_arabic_text(old):
            return True

    messages = [
        {
            "role": "system",
            "content": (
                "You get a list of previous startup interview questions and one "
                "candidate question. Compare them by the exact information or "
                "decision each question requests, not just their broad topic. "
                "Answer YES only if the candidate seeks substantially the same "
                "specific answer as ANY previous question. "
                "Answer NO when it asks for a new detail, an implementation method, "
                "a validation step, evidence, timing, or a decision that no previous "
                "question requested. For example, asking about the price in general "
                "and asking who pays the delivery fee are different questions: NO. "
                "Questions about the same broad subject are not automatically duplicates. "
                "Return exactly YES or NO."
            ),
        },
        {
            "role": "user",
            "content": json.dumps(
                {"previous_questions": previous, "candidate_question": candidate},
                ensure_ascii=False,
            ),
        },
    ]

    result = qwen_generate(messages, max_new_tokens=10, temperature=0.0).strip().upper()
    return result.startswith("YES")


def _updated_shark_summary_v2(session, shark, model_summary=""):
    import re

    model_summary = str(model_summary or "").strip()
    prefix = "ملخص التحليل الحالي:"

    while model_summary.startswith(prefix):
        model_summary = model_summary[len(prefix):].strip()

    # Remove sections that are rebuilt from session data below.
    for marker in ("أحدث إجابات المؤسس:", "نقاط لم تُحسم بعد:"):
        if marker in model_summary:
            model_summary = model_summary.split(marker, 1)[0].strip()

    model_summary = model_summary.rstrip(" |؛").strip()

    if shark != "Mona":
        return (
            prefix + " " + model_summary
            if model_summary
            else "لم يكتمل الملخص بعد؛ ستُحدّثه المقابلة مع ورود معلومات جديدة."
        )

    founder_answers = []
    for item in session.get("history", []):
        if item.get("speaker") == "Founder":
            answer = str(item.get("text", "")).strip()
            if answer and answer not in founder_answers:
                founder_answers.append(answer)

    founder_answers = founder_answers[-5:]

    old_unknowns = _merge_unique_questions(
        [], session.get("unknowns", [])
    )
    
    old_unanswered = _merge_unique_questions(
        [], session.get("unanswered_important_questions", [])
    )
    
    all_questions = _merge_unique_questions(
        old_unknowns,
        old_unanswered,
    )
    
    filtered_questions = _filter_resolved_questions(
        all_questions,
        session.get("history", []),
    )
    
    session["unknowns"] = [
        q for q in filtered_questions
        if q in old_unknowns
    ]
    
    session["unanswered_important_questions"] = [
        q for q in filtered_questions
        if q in old_unanswered
    ]
    
    unknowns = _merge_unique_questions(
        [], session["unknowns"]
    )[-5:]

    parts = []

    if model_summary:
        parts.append(prefix + " " + model_summary)

    if founder_answers:
        parts.append(
            "أحدث إجابات المؤسس: " + "؛ ".join(founder_answers)
        )

    if unknowns:
        parts.append(
            "نقاط لم تُحسم بعد: " + "؛ ".join(unknowns)
        )

    return " | ".join(parts) or (
        "لم تتوفر معلومات كافية لملخص موثوق حتى الآن."
    )

_updated_shark_summary = _updated_shark_summary_v2

print("Interview-flow fix loaded. No server or ngrok was started/restarted.")




In [ ]:
import uuid

jobs = {}
MAX_JOBS_KEPT = 20


def run_analysis_job(job_id, idea, understanding=None):
    try:
        jobs[job_id]["status"] = "running"

        result = analyze_sharks_api(AnalyzeRequest(idea=idea, understanding=understanding))

        # Save the result BEFORE changing the status
        jobs[job_id]["result"] = result
        jobs[job_id]["status"] = "completed"

    except Exception as e:
        jobs[job_id]["error"] = str(e)
        jobs[job_id]["status"] = "failed"


@app.post("/jobs", dependencies=[Depends(check_api_key)])
def create_job(request: AnalyzeRequest):
    # One analysis at a time (one GPU)
    if any(j["status"] in ("queued", "running") for j in jobs.values()):
        raise HTTPException(status_code=429, detail="Another analysis is still running")

    # Keep memory small: drop the oldest jobs
    while len(jobs) >= MAX_JOBS_KEPT:
        jobs.pop(next(iter(jobs)))

    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "queued", "created_at": time.time()}

    thread = threading.Thread(
        target=run_analysis_job,
        args=(job_id, request.idea, request.understanding),
        daemon=True
    )
    thread.start()

    return {"job_id": job_id, "status": "queued"}


@app.get("/jobs/{job_id}", dependencies=[Depends(check_api_key)])
def get_job(job_id: str):
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail="Job not found")

    return jobs[job_id]

In [ ]:
def _is_meta_interview_question(question):
    """Reject questions that discuss the interview instead of the business."""
    normalized = _normalize_question_for_dedup(question or "")
    normalized = normalized.replace("ة", "ه")

    meta_patterns = [
        "نراجع الاسئله السابقه",
        "نحدد ايه المعلومات اللي لسه ناقصه",
        "نراجع المعلومات الناقصه",
        "نحدد المعلومات الناقصه",
        "محتاجين نراجع",
        "خلينا نراجع الاسئله",
    ]

    return any(
        _normalize_question_for_dedup(pattern).replace("ة", "ه")
        in normalized
        for pattern in meta_patterns
    )

In [ ]:
def _filter_resolved_questions(questions, history):
    """Keep unresolved questions; remove only questions the founder clearly answered.
    One model call for the whole list."""
    questions = [str(q or "").strip() for q in (questions or []) if str(q or "").strip()]
    founder_answers = [
        str(item.get("text", "")).strip()
        for item in (history or [])
        if item.get("speaker") == "Founder"
        and str(item.get("text", "")).strip()
    ]

    if not questions or not founder_answers:
        return questions

    numbered = {i: q for i, q in enumerate(questions[-8:], 1)}
    skipped = questions[:-8]

    messages = [
        {
            "role": "system",
            "content": (
                "You get numbered questions and the founder's answers. "
                "Return a JSON array with the numbers of the questions the founder "
                "CLEARLY answered. A partial, vague, or 'I don't know' answer "
                "means the question is NOT answered. "
                "Return only the array, for example [1,3] or []."
            ),
        },
        {
            "role": "user",
            "content": json.dumps(
                {"questions": numbered, "founder_answers": founder_answers[-12:]},
                ensure_ascii=False,
            ),
        },
    ]

    raw = qwen_generate(messages, max_new_tokens=40, temperature=0.0, enable_thinking=False)

    resolved = set()
    match = re.search(r"\[[^\]]*\]", raw)
    if match:
        try:
            resolved = {int(n) for n in json.loads(match.group(0))}
        except (ValueError, TypeError):
            resolved = set()

    return skipped + [q for i, q in numbered.items() if i not in resolved]


In [ ]:
if "server_thread" in globals() and server_thread.is_alive():
    print("Server already running ✅")
else:
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)

    server_thread = threading.Thread(target=server.run, daemon=True)
    server_thread.start()

    # Wait until the server is up (max 20 seconds)
    for _ in range(40):
        if server.started:
            break
        if not server_thread.is_alive():
            raise RuntimeError("Server stopped. Is port 8000 busy? Restart the session.")
        time.sleep(0.5)
    else:
        raise RuntimeError("Server did not start in 20 seconds")

    print("Server running on port 8000 ✅")

In [ ]:
# API client settings (safe; does not send requests or modify sessions)
BASE_URL = "http://127.0.0.1:8000"
headers = {"X-API-Key": JOW_API_KEY, "Content-Type": "application/json"}


In [ ]:
# Reuse an existing tunnel when possible; do not disconnect/create duplicates on rerun.
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
existing_tunnels = ngrok.get_tunnels()
existing_8000 = next(
    (item for item in existing_tunnels if str(item.config.get("addr", "")).endswith(":8000")),
    None,
)
if existing_8000 is not None:
    tunnel = existing_8000
    public_url = tunnel.public_url
    print("Reusing existing ngrok tunnel:", public_url)
else:
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
    print("Created ngrok tunnel:", public_url)
